In [ ]:
from google.colab import files
uploaded = files.upload()

Saving PCOS_LANN_9Feature_Attention.csv to PCOS_LANN_9Feature_Attention.csv


In [ ]:
import pandas as pd
filename = next(iter(uploaded))
df = pd.read_csv(filename)
print("File:", filename)
print("Shape:", df.shape)
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))
print("\nColumns:")
print(df.columns.tolist())

File: PCOS_LANN_9Feature_Attention.csv
Shape: (9, 2)
Number of rows: 9
Number of columns: 2

Columns:
['Feature', 'Attention']


In [ ]:
print("TARGET DISTRIBUTIONS")
for col in ["thyroid_target", "diabetes_target", "pcos_target"]:
    if col in df.columns:
        print("\n", col)
        print(df[col].value_counts(dropna=False))
print("\nDUPLICATE ROWS:", df.duplicated().sum())
if "source_patient_id" in df.columns:
    print("UNIQUE PATIENTS:", df["source_patient_id"].nunique())
    print("TOTAL RECORDS:", len(df))
print("\nDATA TYPES")
print(df.dtypes.value_counts())
print("\nMISSING VALUES — TOP 50")
missing = df.isna().sum().sort_values(ascending=False)
print(missing.head(50))
print("\nMISSING PERCENTAGE — TOP 50")
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print(missing_pct.head(50))

TARGET DISTRIBUTIONS

DUPLICATE ROWS: 0

DATA TYPES
object     1
float64    1
Name: count, dtype: int64

MISSING VALUES — TOP 50
Feature      0
Attention    0
dtype: int64

MISSING PERCENTAGE — TOP 50
Feature      0.0
Attention    0.0
dtype: float64


In [ ]:
print("DISEASE TASK")
print(df["disease_task"].value_counts(dropna=False))
print("\nSOURCE DATASET")
print(df["source_dataset"].value_counts(dropna=False))
print("\nTARGET AVAILABILITY BY DISEASE TASK")
for task in df["disease_task"].dropna().unique():
    temp = df[df["disease_task"] == task]
    print("Disease task:", task)
    print("Rows:", len(temp))
    for target in ["thyroid_target", "diabetes_target", "pcos_target"]:
        if target in temp.columns:
            print(target)
            print(temp[target].value_counts(dropna=False))
print("\nTARGET AVAILABILITY BY SOURCE DATASET")
for source in df["source_dataset"].dropna().unique():
    temp = df[df["source_dataset"] == source]
    print("Source:", source)
    print("Rows:", len(temp))
    for target in ["thyroid_target", "diabetes_target", "pcos_target"]:
        print(target, ":", temp[target].notna().sum(), "non-missing")

DISEASE TASK


KeyError: 'disease_task'

In [ ]:
candidate_features = [
    "age",
    "sex",
    "bmi",
    "height_cm",
    "weight_kg",
    "waist_circumference_cm",
    "hip_circumference_cm",
    "waist_hip_ratio",
    "systolic_bp",
    "diastolic_bp",
    "pulse_rate_bpm",
    "physical_activity_level",
    "regular_exercise",
    "exercise_frequency",
    "exercise_30min_daily",
    "sleep_hours",
    "sleep_7_9h",
    "stress_score",
    "mental_stress",
    "smoking",
    "alcohol",
    "diet_type",
    "dietary_sugar_intake",
    "fast_food",
    "fast_food_regular",
    "balanced_diet",
    "food_quality_good",
    "fresh_food",
    "high_carb_diet",
    "high_meat_intake",
    "meals_on_time",
    "nutritious_food_at_home",
    "packaged_food",
    "vegetarian",
    "water_3l",
    "seasonal_fruits",
    "screen_time_high",
    "emotional_eating",
    "overeating",
    "eat_when_hungry",
    "eat_while_busy",
    "eating_alone",
    "eating_at_home"
]
for disease in ["diabetes", "pcos", "thyroid"]:
    print("\n" + "="*70)
    print(disease.upper())
    print("="*70)
    temp = df[df["disease_task"] == disease]
    result = pd.DataFrame({
        "Available": temp[candidate_features].notna().sum(),
        "Available_%": temp[candidate_features].notna().mean() * 100
    })
    print(result.sort_values("Available_%", ascending=False).to_string())

In [ ]:
check_columns = [
    "sex",
    "physical_activity_level",
    "regular_exercise",
    "exercise_frequency",
    "diet_type",
    "dietary_sugar_intake",
    "fast_food",
    "smoking",
    "alcohol",
    "mental_stress",
    "stress_score",
    "sleep_hours"
]
for col in check_columns:
    print("\n" + "="*50)
    print(col)
    print("="*50)
    print(df[col].value_counts(dropna=False).head(20))

In [ ]:
numeric_features = [
    "age",
    "bmi",
    "height_cm",
    "weight_kg",
    "waist_circumference_cm",
    "hip_circumference_cm",
    "waist_hip_ratio",
    "systolic_bp",
    "diastolic_bp",
    "pulse_rate_bpm",
    "sleep_hours",
    "stress_score"
]
print(df[numeric_features].describe().T.to_string())

In [ ]:
diabetes_df = df[df["disease_task"] == "diabetes"].copy()
print("Diabetes dataset shape:", diabetes_df.shape)
print("\nTarget distribution:")
print(diabetes_df["diabetes_target"].value_counts())
print("\nMissing target:")
print(diabetes_df["diabetes_target"].isna().sum())

In [ ]:
diabetes_features = [
    "age",
    "bmi",
    "sex",
    "smoking",
    "sleep_hours",
    "diet_type",
    "stress_score",
    "exercise_frequency",
    "alcohol"
]
X = diabetes_df[diabetes_features].copy()
y = diabetes_df["diabetes_target"].astype(int)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nMissing values:")
print(X.isna().sum())
print("\nTarget:")
print(y.value_counts())

In [ ]:
X.loc[(X["age"] < 18) | (X["age"] > 100), "age"] = None
X["sex"] = X["sex"].replace({
    "F": "Female",
    "Female": "Female",
    "M": "Male",
    "Male": "Male"
})
print("Age range:")
print(X["age"].min(), "to", X["age"].max())
print("\nSex values:")
print(X["sex"].value_counts(dropna=False))

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)
print("\nTraining target:")
print(y_train.value_counts())
print("\nValidation target:")
print(y_val.value_counts())
print("\nTesting target:")
print(y_test.value_counts())

In [ ]:
numeric_features = [
    "age",
    "bmi",
    "sleep_hours",
    "stress_score"
]
categorical_features = [
    "sex",
    "smoking",
    "diet_type",
    "exercise_frequency",
    "alcohol"
]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)
print("Training processed shape:", X_train_processed.shape)
print("Validation processed shape:", X_val_processed.shape)
print("Testing processed shape:", X_test_processed.shape)

In [ ]:
X_train_processed = X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else X_train_processed
X_val_processed = X_val_processed.toarray() if hasattr(X_val_processed, "toarray") else X_val_processed
X_test_processed = X_test_processed.toarray() if hasattr(X_test_processed, "toarray") else X_test_processed
print("Final training shape:", X_train_processed.shape)
print("Final validation shape:", X_val_processed.shape)
print("Final testing shape:", X_test_processed.shape)

In [ ]:
print("Training:")
print(y_train.value_counts())
print("\nValidation:")
print(y_val.value_counts())
print("\nTesting:")
print(y_test.value_counts())

In [ ]:
print("\nShapes:")
print("X_train:", X_train_processed.shape)
print("X_val:", X_val_processed.shape)
print("X_test:", X_test_processed.shape)
print("\nTarget shapes:")
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)
lr_model.fit(X_train_processed, y_train)
print("Logistic Regression trained successfully!")

In [ ]:
y_val_pred_lr = lr_model.predict(X_val_processed)
y_val_prob_lr = lr_model.predict_proba(X_val_processed)[:, 1]
print("Validation predictions generated!")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
lr_accuracy = accuracy_score(y_val, y_val_pred_lr)
lr_precision = precision_score(y_val, y_val_pred_lr)
lr_recall = recall_score(y_val, y_val_pred_lr)
lr_f1 = f1_score(y_val, y_val_pred_lr)
lr_roc_auc = roc_auc_score(y_val, y_val_prob_lr)
print("Logistic Regression - Validation Results")
print("Accuracy :", round(lr_accuracy, 4))
print("Precision:", round(lr_precision, 4))
print("Recall   :", round(lr_recall, 4))
print("F1 Score :", round(lr_f1, 4))
print("ROC-AUC  :", round(lr_roc_auc, 4))

In [ ]:
from sklearn.metrics import confusion_matrix
cm_lr = confusion_matrix(y_val, y_val_pred_lr)
print("Confusion Matrix:")
print(cm_lr)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay(
    confusion_matrix=cm_lr,
    display_labels=["No Diabetes", "Diabetes"]
).plot()
plt.title("Logistic Regression - Diabetes Confusion Matrix")
plt.show()

In [ ]:
from sklearn.tree import DecisionTreeClassifier
dt_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)
dt_model.fit(X_train_processed, y_train)
print("Decision Tree trained successfully!")

In [ ]:
y_val_pred_dt = dt_model.predict(X_val_processed)
y_val_prob_dt = dt_model.predict_proba(X_val_processed)[:, 1]
print("Decision Tree predictions generated!")

In [ ]:
dt_accuracy = accuracy_score(y_val, y_val_pred_dt)
dt_precision = precision_score(y_val, y_val_pred_dt)
dt_recall = recall_score(y_val, y_val_pred_dt)
dt_f1 = f1_score(y_val, y_val_pred_dt)
dt_roc_auc = roc_auc_score(y_val, y_val_prob_dt)
print("Decision Tree - Validation Results")
print("Accuracy :", round(dt_accuracy, 4))
print("Precision:", round(dt_precision, 4))
print("Recall   :", round(dt_recall, 4))
print("F1 Score :", round(dt_f1, 4))
print("ROC-AUC  :", round(dt_roc_auc, 4))

In [ ]:
cm_dt = confusion_matrix(y_val, y_val_pred_dt)
print("Decision Tree Confusion Matrix:")
print(cm_dt)

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix=cm_dt,
    display_labels=["No Diabetes", "Diabetes"]
).plot()
plt.title("Decision Tree - Diabetes Confusion Matrix")
plt.show()

In [ ]:
import pandas as pd
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree"
    ],
    "Accuracy": [
        lr_accuracy,
        dt_accuracy
    ],
    "Precision": [
        lr_precision,
        dt_precision
    ],
    "Recall": [
        lr_recall,
        dt_recall
    ],
    "F1 Score": [
        lr_f1,
        dt_f1
    ],
    "ROC-AUC": [
        lr_roc_auc,
        dt_roc_auc
    ]
})
results = results.round(4)
print(results)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_processed, y_train)
print("Random Forest trained successfully!")

In [ ]:
y_val_pred_rf = rf_model.predict(X_val_processed)
y_val_prob_rf = rf_model.predict_proba(X_val_processed)[:, 1]
print("Random Forest predictions generated!")

In [ ]:
rf_accuracy = accuracy_score(y_val, y_val_pred_rf)
rf_precision = precision_score(y_val, y_val_pred_rf)
rf_recall = recall_score(y_val, y_val_pred_rf)
rf_f1 = f1_score(y_val, y_val_pred_rf)
rf_roc_auc = roc_auc_score(y_val, y_val_prob_rf)
print("Random Forest - Validation Results")
print("Accuracy :", round(rf_accuracy, 4))
print("Precision:", round(rf_precision, 4))
print("Recall   :", round(rf_recall, 4))
print("F1 Score :", round(rf_f1, 4))
print("ROC-AUC  :", round(rf_roc_auc, 4))

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "Accuracy": [
        lr_accuracy,
        dt_accuracy,
        rf_accuracy
    ],
    "Precision": [
        lr_precision,
        dt_precision,
        rf_precision
    ],
    "Recall": [
        lr_recall,
        dt_recall,
        rf_recall
    ],
    "F1 Score": [
        lr_f1,
        dt_f1,
        rf_f1
    ],
    "ROC-AUC": [
        lr_roc_auc,
        dt_roc_auc,
        rf_roc_auc
    ]
})
results = results.round(4)
print(results)

In [ ]:
!pip install -q xgboost
from xgboost import XGBClassifier
print("XGBoost imported successfully!")

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)
xgb_model.fit(
    X_train_processed,
    y_train
)
print("XGBoost trained successfully!")

In [ ]:
y_val_pred_xgb = xgb_model.predict(X_val_processed)
y_val_prob_xgb = xgb_model.predict_proba(X_val_processed)[:, 1]
print("XGBoost predictions generated!")

In [ ]:
xgb_accuracy = accuracy_score(y_val, y_val_pred_xgb)
xgb_precision = precision_score(y_val, y_val_pred_xgb)
xgb_recall = recall_score(y_val, y_val_pred_xgb)
xgb_f1 = f1_score(y_val, y_val_pred_xgb)
xgb_roc_auc = roc_auc_score(y_val, y_val_prob_xgb)
print("XGBoost - Validation Results")
print("Accuracy :", round(xgb_accuracy, 4))
print("Precision:", round(xgb_precision, 4))
print("Recall   :", round(xgb_recall, 4))
print("F1 Score :", round(xgb_f1, 4))
print("ROC-AUC  :", round(xgb_roc_auc, 4))

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        lr_accuracy,
        dt_accuracy,
        rf_accuracy,
        xgb_accuracy
    ],
    "Precision": [
        lr_precision,
        dt_precision,
        rf_precision,
        xgb_precision
    ],
    "Recall": [
        lr_recall,
        dt_recall,
        rf_recall,
        xgb_recall
    ],
    "F1 Score": [
        lr_f1,
        dt_f1,
        rf_f1,
        xgb_f1
    ],
    "ROC-AUC": [
        lr_roc_auc,
        dt_roc_auc,
        rf_roc_auc,
        xgb_roc_auc
    ]
})
results = results.round(4)
print(results)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
print("TensorFlow version:", tf.__version__)

In [ ]:
dnn_model = Sequential([
    Dense(64, activation="relu", input_shape=(X_train_processed.shape[1],)),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])
dnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
dnn_model.summary()

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)
history_dnn = dnn_model.fit(
    X_train_processed,
    y_train,
    validation_data=(X_val_processed, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
y_val_prob_dnn = dnn_model.predict(
    X_val_processed,
    verbose=0
).ravel()
y_val_pred_dnn = (y_val_prob_dnn >= 0.5).astype(int)
print("DNN predictions generated!")

In [ ]:
dnn_accuracy = accuracy_score(y_val, y_val_pred_dnn)
dnn_precision = precision_score(y_val, y_val_pred_dnn)
dnn_recall = recall_score(y_val, y_val_pred_dnn)
dnn_f1 = f1_score(y_val, y_val_pred_dnn)
dnn_roc_auc = roc_auc_score(y_val, y_val_prob_dnn)
print("Standard DNN - Validation Results")
print("Accuracy :", round(dnn_accuracy, 4))
print("Precision:", round(dnn_precision, 4))
print("Recall   :", round(dnn_recall, 4))
print("F1 Score :", round(dnn_f1, 4))
print("ROC-AUC  :", round(dnn_roc_auc, 4))

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost",
        "Standard DNN"
    ],
    "Accuracy": [
        lr_accuracy,
        dt_accuracy,
        rf_accuracy,
        xgb_accuracy,
        dnn_accuracy
    ],
    "Precision": [
        lr_precision,
        dt_precision,
        rf_precision,
        xgb_precision,
        dnn_precision
    ],
    "Recall": [
        lr_recall,
        dt_recall,
        rf_recall,
        xgb_recall,
        dnn_recall
    ],
    "F1 Score": [
        lr_f1,
        dt_f1,
        rf_f1,
        xgb_f1,
        dnn_f1
    ],
    "ROC-AUC": [
        lr_roc_auc,
        dt_roc_auc,
        rf_roc_auc,
        xgb_roc_auc,
        dnn_roc_auc
    ]
})
results = results.round(4)
print(results)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_dnn.history["loss"], label="Training Loss")
plt.plot(history_dnn.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Standard DNN Training and Validation Loss")
plt.legend()
plt.show()

In [ ]:
from tensorflow.keras.layers import Input, Multiply
from tensorflow.keras.models import Model
print("LANN layers imported successfully!")

In [ ]:
input_dim = X_train_processed.shape[1]
inputs = Input(shape=(input_dim,))
attention_scores = Dense(
    input_dim,
    activation="softmax",
    name="feature_attention"
)(inputs)
attended_features = Multiply()([
    inputs,
    attention_scores
])
x = Dense(64, activation="relu")(attended_features)
x = Dropout(0.2)(x)
x = Dense(32, activation="relu")(x)
x = Dropout(0.2)(x)
outputs = Dense(1, activation="sigmoid")(x)
lann_model = Model(
    inputs=inputs,
    outputs=outputs,
    name="Lifestyle_Attention_Neural_Network"
)
lann_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
lann_model.summary()

In [ ]:
early_stopping_lann = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)
history_lann = lann_model.fit(
    X_train_processed,
    y_train,
    validation_data=(X_val_processed, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping_lann],
    verbose=1
)

In [ ]:
y_val_prob_lann = lann_model.predict(
    X_val_processed,
    verbose=0
).ravel()
y_val_pred_lann = (y_val_prob_lann >= 0.5).astype(int)
print("LANN predictions generated!")

In [ ]:
lann_accuracy = accuracy_score(y_val, y_val_pred_lann)
lann_precision = precision_score(y_val, y_val_pred_lann)
lann_recall = recall_score(y_val, y_val_pred_lann)
lann_f1 = f1_score(y_val, y_val_pred_lann)
lann_roc_auc = roc_auc_score(y_val, y_val_prob_lann)
print("LANN - Validation Results")
print("Accuracy :", round(lann_accuracy, 4))
print("Precision:", round(lann_precision, 4))
print("Recall   :", round(lann_recall, 4))
print("F1 Score :", round(lann_f1, 4))
print("ROC-AUC  :", round(lann_roc_auc, 4))

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost",
        "Standard DNN",
        "LANN"
    ],
    "Accuracy": [
        lr_accuracy,
        dt_accuracy,
        rf_accuracy,
        xgb_accuracy,
        dnn_accuracy,
        lann_accuracy
    ],
    "Precision": [
        lr_precision,
        dt_precision,
        rf_precision,
        xgb_precision,
        dnn_precision,
        lann_precision
    ],
    "Recall": [
        lr_recall,
        dt_recall,
        rf_recall,
        xgb_recall,
        dnn_recall,
        lann_recall
    ],
    "F1 Score": [
        lr_f1,
        dt_f1,
        rf_f1,
        xgb_f1,
        dnn_f1,
        lann_f1
    ],
    "ROC-AUC": [
        lr_roc_auc,
        dt_roc_auc,
        rf_roc_auc,
        xgb_roc_auc,
        dnn_roc_auc,
        lann_roc_auc
    ]
})
results = results.round(4)
print(results)

In [ ]:
cm_lann = confusion_matrix(y_val, y_val_pred_lann)
print("LANN Confusion Matrix:")
print(cm_lann)

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix=cm_lann,
    display_labels=["No Diabetes", "Diabetes"]
).plot()
plt.title("LANN - Diabetes Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import roc_curve
fpr_lr, tpr_lr, _ = roc_curve(y_val, y_val_prob_lr)
fpr_dt, tpr_dt, _ = roc_curve(y_val, y_val_prob_dt)
fpr_rf, tpr_rf, _ = roc_curve(y_val, y_val_prob_rf)
fpr_xgb, tpr_xgb, _ = roc_curve(y_val, y_val_prob_xgb)
fpr_dnn, tpr_dnn, _ = roc_curve(y_val, y_val_prob_dnn)
fpr_lann, tpr_lann, _ = roc_curve(y_val, y_val_prob_lann)
plt.figure(figsize=(9, 7))
plt.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC = {lr_roc_auc:.3f})")
plt.plot(fpr_dt, tpr_dt, label=f"Decision Tree (AUC = {dt_roc_auc:.3f})")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {rf_roc_auc:.3f})")
plt.plot(fpr_xgb, tpr_xgb, label=f"XGBoost (AUC = {xgb_roc_auc:.3f})")
plt.plot(fpr_dnn, tpr_dnn, label=f"Standard DNN (AUC = {dnn_roc_auc:.3f})")
plt.plot(fpr_lann, tpr_lann, label=f"LANN (AUC = {lann_roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison - Diabetes")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    history_lann.history["loss"],
    label="Training Loss"
)
plt.plot(
    history_lann.history["val_loss"],
    label="Validation Loss"
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("LANN Training and Validation Loss")
plt.legend()
plt.grid()
plt.show()

In [ ]:
attention_layer = lann_model.get_layer("feature_attention")
print(attention_layer)

In [ ]:
attention_model = Model(
    inputs=lann_model.input,
    outputs=attention_layer.output
)
print("Attention model created successfully!")

In [ ]:
attention_weights = attention_model.predict(
    X_val_processed,
    verbose=0
)
print("Attention weights shape:", attention_weights.shape)

In [ ]:
feature_names = preprocessor.get_feature_names_out()
print("Number of feature names:", len(feature_names))
for i, name in enumerate(feature_names):
    print(i, name)

In [ ]:
import numpy as np
import pandas as pd
mean_attention = attention_weights.mean(axis=0)
attention_df = pd.DataFrame({
    "Feature": feature_names,
    "Mean_Attention": mean_attention
})
attention_df = attention_df.sort_values(
    "Mean_Attention",
    ascending=False
).reset_index(drop=True)
attention_df["Mean_Attention"] = attention_df["Mean_Attention"].round(4)
print(attention_df)

In [ ]:
plt.figure(figsize=(10, 7))
plt.barh(
    attention_df["Feature"],
    attention_df["Mean_Attention"]
)
plt.xlabel("Mean Attention Weight")
plt.ylabel("Feature")
plt.title("LANN Feature Attention - Diabetes")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
!pip install -q shap
import shap
print("SHAP version:", shap.__version__)

In [ ]:
explainer_lr = shap.LinearExplainer(
    lr_model,
    X_train_processed
)
shap_values_lr = explainer_lr(
    X_val_processed
)
print("SHAP values generated!")

In [ ]:
shap.summary_plot(
    shap_values_lr,
    X_val_processed,
    feature_names=feature_names
)

In [ ]:
print(diabetes_df["alcohol"].value_counts(dropna=False))

In [ ]:
print(diabetes_df[["alcohol", "source_dataset"]].dropna(
    subset=["alcohol"]
).groupby(["source_dataset", "alcohol"]).size())

In [ ]:
print(diabetes_df[diabetes_df["source_dataset"] == "kaggle_diabetes_lifestyle_189"][
    ["alcohol", "source_record_id", "source_patient_id"]
].head(20))
print("\nAlcohol 0 rows:")
print(diabetes_df[
    (diabetes_df["source_dataset"] == "kaggle_diabetes_lifestyle_189") &
    (diabetes_df["alcohol"] == 0)
][["alcohol", "age", "bmi", "smoking", "diet_type"]].head(10))
print("\nAlcohol 1 rows:")
print(diabetes_df[
    (diabetes_df["source_dataset"] == "kaggle_diabetes_lifestyle_189") &
    (diabetes_df["alcohol"] == 1)
][["alcohol", "age", "bmi", "smoking", "diet_type"]].head(10))

In [ ]:
print("Source datasets:")
print(diabetes_df["source_dataset"].value_counts(dropna=False))
print("\nAlcohol dtype:")
print(diabetes_df["alcohol"].dtype)

print("\n189 dataset alcohol values:")
temp = diabetes_df[
    diabetes_df["source_dataset"].astype(str).str.strip() == "kaggle_diabetes_lifestyle_189"
].copy()

print(temp[["alcohol", "age", "bmi", "smoking", "diet_type"]].to_string(index=False))

print("\nUnique alcohol values in 189 dataset:")
print(temp["alcohol"].value_counts(dropna=False))

In [ ]:
diabetes_features = [
    "age",
    "bmi",
    "sex",
    "smoking",
    "sleep_hours",
    "diet_type",
    "stress_score",
    "exercise_frequency"
]

X = diabetes_df[diabetes_features].copy()
y = diabetes_df["diabetes_target"].astype(int)

print("Features:", diabetes_features)
print("X shape:", X.shape)
print("Missing values:")
print(X.isna().sum())

In [ ]:
patient_counts = diabetes_df["source_patient_id"].value_counts(dropna=False)

print("Total rows:", len(diabetes_df))
print("Unique patient IDs:", diabetes_df["source_patient_id"].nunique(dropna=True))
print("Duplicated patient IDs:", (patient_counts > 1).sum())
print("Maximum rows for one patient:", patient_counts.max())

print("\nPatients appearing more than once:")
print(patient_counts[patient_counts > 1].head(20))

In [ ]:
from sklearn.model_selection import train_test_split
patient_data = diabetes_df[["source_patient_id", "diabetes_target"]].drop_duplicates()

print("Unique patients:", len(patient_data))
print("Patients per class:")
print(patient_data["diabetes_target"].value_counts())

train_patients, temp_patients = train_test_split(
    patient_data,
    test_size=0.30,
    random_state=42,
    stratify=patient_data["diabetes_target"]
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=42,
    stratify=temp_patients["diabetes_target"]
)

train_patient_ids = set(train_patients["source_patient_id"])
val_patient_ids = set(val_patients["source_patient_id"])
test_patient_ids = set(test_patients["source_patient_id"])

train_mask = diabetes_df["source_patient_id"].isin(train_patient_ids)
val_mask = diabetes_df["source_patient_id"].isin(val_patient_ids)
test_mask = diabetes_df["source_patient_id"].isin(test_patient_ids)

X_train = X.loc[train_mask].copy()
X_val = X.loc[val_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_val = y.loc[val_mask].copy()
y_test = y.loc[test_mask].copy()

print("\nDataset shapes:")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nTarget distribution:")
print("Train:")
print(y_train.value_counts())

print("\nValidation:")
print(y_val.value_counts())

print("\nTest:")
print(y_test.value_counts())

In [ ]:
train_ids = set(diabetes_df.loc[train_mask, "source_patient_id"])
val_ids = set(diabetes_df.loc[val_mask, "source_patient_id"])
test_ids = set(diabetes_df.loc[test_mask, "source_patient_id"])

print("Train ∩ Validation:", len(train_ids & val_ids))
print("Train ∩ Test:", len(train_ids & test_ids))
print("Validation ∩ Test:", len(val_ids & test_ids))

In [ ]:
target_consistency = (
    diabetes_df
    .groupby("source_patient_id")["diabetes_target"]
    .nunique()
)

print("Patients with conflicting targets:",
      (target_consistency > 1).sum())

print(
    diabetes_df[
        diabetes_df["source_patient_id"].isin(
            target_consistency[target_consistency > 1].index
        )
    ][["source_patient_id", "diabetes_target"]]
)

In [ ]:
diabetes_df["patient_key"] = (
    diabetes_df["source_dataset"].astype(str)
    + "_"
    + diabetes_df["source_patient_id"].astype(str)
)

print("Total rows:", len(diabetes_df))
print("Unique patient keys:", diabetes_df["patient_key"].nunique())

print("\nDuplicated patient keys:")
patient_key_counts = diabetes_df["patient_key"].value_counts()
print(patient_key_counts[patient_key_counts > 1])

print("\nMaximum rows per patient key:",
      patient_key_counts.max())

In [ ]:
conflict_check = (
    diabetes_df
    .groupby("patient_key")["diabetes_target"]
    .nunique()
)

print("Patient keys with conflicting targets:",
      (conflict_check > 1).sum())

if (conflict_check > 1).sum() > 0:
    print(
        diabetes_df[
            diabetes_df["patient_key"].isin(
                conflict_check[conflict_check > 1].index
            )
        ][[
            "patient_key",
            "source_dataset",
            "source_patient_id",
            "diabetes_target"
        ]]
    )

In [ ]:
from sklearn.model_selection import train_test_split

patient_data = diabetes_df[
    ["patient_key", "diabetes_target"]
].copy()

# One row per patient key
patient_data = patient_data.drop_duplicates("patient_key")

train_patients, temp_patients = train_test_split(
    patient_data,
    test_size=0.30,
    random_state=42,
    stratify=patient_data["diabetes_target"]
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=42,
    stratify=temp_patients["diabetes_target"]
)

train_patient_keys = set(train_patients["patient_key"])
val_patient_keys = set(val_patients["patient_key"])
test_patient_keys = set(test_patients["patient_key"])

train_mask = diabetes_df["patient_key"].isin(train_patient_keys)
val_mask = diabetes_df["patient_key"].isin(val_patient_keys)
test_mask = diabetes_df["patient_key"].isin(test_patient_keys)

X_train = X.loc[train_mask].copy()
X_val = X.loc[val_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_val = y.loc[val_mask].copy()
y_test = y.loc[test_mask].copy()

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nTrain targets:")
print(y_train.value_counts())

print("\nValidation targets:")
print(y_val.value_counts())

print("\nTest targets:")
print(y_test.value_counts())

In [ ]:
print("Train ∩ Validation:",
      len(train_patient_keys & val_patient_keys))

print("Train ∩ Test:",
      len(train_patient_keys & test_patient_keys))

print("Validation ∩ Test:",
      len(val_patient_keys & test_patient_keys))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
diabetes_df["patient_key"] = (
    diabetes_df["source_dataset"].astype(str)
    + "_"
    + diabetes_df["source_patient_id"].astype(str)
)

print("Total rows:", len(diabetes_df))
print("Unique patient keys:", diabetes_df["patient_key"].nunique())

patient_key_counts = diabetes_df["patient_key"].value_counts()

print("\nDuplicated patient keys:")
print(patient_key_counts[patient_key_counts > 1])

print("\nMaximum rows per patient key:",
      patient_key_counts.max())

patient_data = diabetes_df[
    ["patient_key", "diabetes_target"]
].drop_duplicates("patient_key")

print("\nUnique patients:", len(patient_data))
print("\nPatient target distribution:")
print(patient_data["diabetes_target"].value_counts())

train_patients, temp_patients = train_test_split(
    patient_data,
    test_size=0.30,
    random_state=42,
    stratify=patient_data["diabetes_target"]
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=42,
    stratify=temp_patients["diabetes_target"]
)

train_patient_keys = set(train_patients["patient_key"])
val_patient_keys = set(val_patients["patient_key"])
test_patient_keys = set(test_patients["patient_key"])

train_mask = diabetes_df["patient_key"].isin(train_patient_keys)
val_mask = diabetes_df["patient_key"].isin(val_patient_keys)
test_mask = diabetes_df["patient_key"].isin(test_patient_keys)

X_train = X.loc[train_mask].copy()
X_val = X.loc[val_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_val = y.loc[val_mask].copy()
y_test = y.loc[test_mask].copy()

print("\nDataset shapes:")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nTarget distribution:")
print("Train:")
print(y_train.value_counts())

print("\nValidation:")
print(y_val.value_counts())

print("\nTest:")
print(y_test.value_counts())

print("\nPatient leakage check:")
print("Train ∩ Validation:",
      len(train_patient_keys & val_patient_keys))

print("Train ∩ Test:",
      len(train_patient_keys & test_patient_keys))

print("Validation ∩ Test:",
      len(val_patient_keys & test_patient_keys))

numeric_features = [
    "age",
    "bmi",
    "sleep_hours",
    "stress_score"
]

categorical_features = [
    "sex",
    "smoking",
    "diet_type",
    "exercise_frequency"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

X_train_processed = (
    X_train_processed.toarray()
    if hasattr(X_train_processed, "toarray")
    else X_train_processed
)

X_val_processed = (
    X_val_processed.toarray()
    if hasattr(X_val_processed, "toarray")
    else X_val_processed
)

X_test_processed = (
    X_test_processed.toarray()
    if hasattr(X_test_processed, "toarray")
    else X_test_processed
)

print("\nProcessed shapes:")
print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)

feature_names = preprocessor.get_feature_names_out()

print("\nNumber of processed features:", len(feature_names))

print("\nProcessed feature names:")
for i, feature in enumerate(feature_names):
    print(i, feature)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train_processed, y_train)

y_val_pred_lr = lr_model.predict(X_val_processed)
y_val_prob_lr = lr_model.predict_proba(X_val_processed)[:, 1]

lr_results = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_val, y_val_pred_lr),
    "Precision": precision_score(y_val, y_val_pred_lr),
    "Recall": recall_score(y_val, y_val_pred_lr),
    "F1": f1_score(y_val, y_val_pred_lr),
    "ROC-AUC": roc_auc_score(y_val, y_val_prob_lr)
}

print(lr_results)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_lr))

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

dt_model.fit(X_train_processed, y_train)

y_val_pred_dt = dt_model.predict(X_val_processed)
y_val_prob_dt = dt_model.predict_proba(X_val_processed)[:, 1]

dt_results = {
    "Model": "Decision Tree",
    "Accuracy": accuracy_score(y_val, y_val_pred_dt),
    "Precision": precision_score(y_val, y_val_pred_dt),
    "Recall": recall_score(y_val, y_val_pred_dt),
    "F1": f1_score(y_val, y_val_pred_dt),
    "ROC-AUC": roc_auc_score(y_val, y_val_prob_dt)
}

print(dt_results)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_dt))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_processed, y_train)

y_val_pred_rf = rf_model.predict(X_val_processed)
y_val_prob_rf = rf_model.predict_proba(X_val_processed)[:, 1]

rf_results = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_val, y_val_pred_rf),
    "Precision": precision_score(y_val, y_val_pred_rf),
    "Recall": recall_score(y_val, y_val_pred_rf),
    "F1": f1_score(y_val, y_val_pred_rf),
    "ROC-AUC": roc_auc_score(y_val, y_val_prob_rf)
}

print(rf_results)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_rf))

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(X_train_processed, y_train)

y_val_pred_xgb = xgb_model.predict(X_val_processed)
y_val_prob_xgb = xgb_model.predict_proba(X_val_processed)[:, 1]

xgb_results = {
    "Model": "XGBoost",
    "Accuracy": accuracy_score(y_val, y_val_pred_xgb),
    "Precision": precision_score(y_val, y_val_pred_xgb),
    "Recall": recall_score(y_val, y_val_pred_xgb),
    "F1": f1_score(y_val, y_val_pred_xgb),
    "ROC-AUC": roc_auc_score(y_val, y_val_prob_xgb)
}

print(xgb_results)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_xgb))

In [ ]:
baseline_results = pd.DataFrame([
    lr_results,
    dt_results,
    rf_results,
    xgb_results
])

print(baseline_results.round(4))

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)

dnn_model = Sequential([
    Input(shape=(X_train_processed.shape[1],)),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

dnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping_dnn = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history_dnn = dnn_model.fit(
    X_train_processed,
    y_train,
    validation_data=(X_val_processed, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping_dnn],
    verbose=1
)

In [ ]:
y_val_prob_dnn = dnn_model.predict(
    X_val_processed,
    verbose=0
).ravel()

y_val_pred_dnn = (y_val_prob_dnn >= 0.5).astype(int)

dnn_results = {
    "Model": "Standard DNN",
    "Accuracy": accuracy_score(y_val, y_val_pred_dnn),
    "Precision": precision_score(y_val, y_val_pred_dnn),
    "Recall": recall_score(y_val, y_val_pred_dnn),
    "F1": f1_score(y_val, y_val_pred_dnn),
    "ROC-AUC": roc_auc_score(y_val, y_val_prob_dnn)
}

print(dnn_results)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_dnn))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(history_dnn.history["accuracy"], label="Training Accuracy")
plt.plot(history_dnn.history["val_accuracy"], label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Standard DNN Training and Validation Accuracy")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(history_dnn.history["loss"], label="Training Loss")
plt.plot(history_dnn.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Standard DNN Training and Validation Loss")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, Multiply

tf.random.set_seed(42)

input_dim = X_train_processed.shape[1]

inputs = Input(shape=(input_dim,))

# Feature attention
attention_scores = Dense(
    input_dim,
    activation="softmax",
    name="feature_attention"
)(inputs)

# Apply attention weights to input features
attended_features = Multiply()([
    inputs,
    attention_scores
])

# Neural network
x = Dense(64, activation="relu")(attended_features)
x = Dropout(0.2)(x)

x = Dense(32, activation="relu")(x)
x = Dropout(0.2)(x)

outputs = Dense(1, activation="sigmoid")(x)

lann_model = Model(
    inputs=inputs,
    outputs=outputs,
    name="Lifestyle_Attention_Neural_Network"
)

lann_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lann_model.summary()

In [ ]:
early_stopping_lann = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history_lann = lann_model.fit(
    X_train_processed,
    y_train,
    validation_data=(X_val_processed, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping_lann],
    verbose=1
)

In [ ]:
y_val_prob_lann = lann_model.predict(
    X_val_processed,
    verbose=0
).ravel()

y_val_pred_lann = (
    y_val_prob_lann >= 0.5
).astype(int)

lann_results = {
    "Model": "LANN",
    "Accuracy": accuracy_score(y_val, y_val_pred_lann),
    "Precision": precision_score(y_val, y_val_pred_lann),
    "Recall": recall_score(y_val, y_val_pred_lann),
    "F1": f1_score(y_val, y_val_pred_lann),
    "ROC-AUC": roc_auc_score(y_val, y_val_prob_lann)
}

print(lann_results)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_lann))

In [ ]:
all_validation_results = pd.DataFrame([
    lr_results,
    dt_results,
    rf_results,
    xgb_results,
    dnn_results,
    lann_results
])

print(
    all_validation_results
    .round(4)
    .to_string(index=False)
)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history_lann.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history_lann.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("LANN Training and Validation Accuracy")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history_lann.history["loss"],
    label="Training Loss"
)

plt.plot(
    history_lann.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("LANN Training and Validation Loss")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
attention_layer = lann_model.get_layer("feature_attention")

attention_model = Model(
    inputs=lann_model.input,
    outputs=attention_layer.output
)

attention_weights = attention_model.predict(
    X_val_processed,
    verbose=0
)

mean_attention = attention_weights.mean(axis=0)

attention_df = pd.DataFrame({
    "Feature": feature_names,
    "Mean_Attention": mean_attention
})

attention_df = attention_df.sort_values(
    "Mean_Attention",
    ascending=False
).reset_index(drop=True)

print(attention_df.to_string(index=False))

In [ ]:
top_attention = attention_df.head(10).sort_values(
    "Mean_Attention"
)

plt.figure(figsize=(9, 6))

plt.barh(
    top_attention["Feature"],
    top_attention["Mean_Attention"]
)

plt.xlabel("Mean Attention Weight")
plt.ylabel("Feature")
plt.title("Top 10 LANN Attention Weights")

plt.tight_layout()
plt.show()

In [ ]:
!pip install -q shap
import shap

print("SHAP version:", shap.__version__)

In [ ]:
explainer_lr = shap.LinearExplainer(
    lr_model,
    X_train_processed
)

shap_values_lr = explainer_lr(
    X_val_processed
)

print("SHAP values shape:", shap_values_lr.values.shape)

In [ ]:
plt.figure(figsize=(10, 7))

shap.summary_plot(
    shap_values_lr.values,
    X_val_processed,
    feature_names=feature_names,
    show=False
)

plt.title("SHAP Feature Importance - Logistic Regression")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))

shap.summary_plot(
    shap_values_lr.values,
    X_val_processed,
    feature_names=feature_names,
    plot_type="bar",
    show=False
)

plt.title("Mean Absolute SHAP Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
background_size = min(100, len(X_train_processed))

background = X_train_processed[
    :background_size
]

explainer_lann = shap.Explainer(
    lann_model,
    background
)

shap_values_lann = explainer_lann(
    X_val_processed
)

print("SHAP values shape:")

if hasattr(shap_values_lann, "values"):
    print(shap_values_lann.values.shape)
else:
    print("SHAP values generated")

In [ ]:
shap.summary_plot(
    shap_values_lann.values,
    X_val_processed,
    feature_names=feature_names
)

In [ ]:
shap.summary_plot(
    shap_values_lann.values,
    X_val_processed,
    feature_names=feature_names,
    plot_type="bar"
)

In [ ]:
import seaborn as sns

cm_lann = confusion_matrix(
    y_val,
    y_val_pred_lann
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_lann,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Diabetes", "Diabetes"],
    yticklabels=["No Diabetes", "Diabetes"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("LANN Confusion Matrix")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve

models_probabilities = {
    "Logistic Regression": y_val_prob_lr,
    "Decision Tree": y_val_prob_dt,
    "Random Forest": y_val_prob_rf,
    "XGBoost": y_val_prob_xgb,
    "Standard DNN": y_val_prob_dnn,
    "LANN": y_val_prob_lann
}

plt.figure(figsize=(9, 7))

for model_name, probabilities in models_probabilities.items():

    fpr, tpr, _ = roc_curve(
        y_val,
        probabilities
    )

    auc_score = roc_auc_score(
        y_val,
        probabilities
    )

    plt.plot(
        fpr,
        tpr,
        label=f"{model_name} (AUC = {auc_score:.3f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves - Diabetes Classification")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.show()

In [ ]:

y_test_pred_lr = lr_model.predict(X_test_processed)
y_test_prob_lr = lr_model.predict_proba(X_test_processed)[:, 1]

y_test_pred_dt = dt_model.predict(X_test_processed)
y_test_prob_dt = dt_model.predict_proba(X_test_processed)[:, 1]

y_test_pred_rf = rf_model.predict(X_test_processed)
y_test_prob_rf = rf_model.predict_proba(X_test_processed)[:, 1]

y_test_pred_xgb = xgb_model.predict(X_test_processed)
y_test_prob_xgb = xgb_model.predict_proba(X_test_processed)[:, 1]

y_test_prob_dnn = dnn_model.predict(
    X_test_processed,
    verbose=0
).ravel()

y_test_pred_dnn = (
    y_test_prob_dnn >= 0.5
).astype(int)

y_test_prob_lann = lann_model.predict(
    X_test_processed,
    verbose=0
).ravel()

y_test_pred_lann = (
    y_test_prob_lann >= 0.5
).astype(int)


def calculate_metrics(y_true, y_pred, y_prob, model_name):

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob)
    }


test_results = pd.DataFrame([
    calculate_metrics(
        y_test,
        y_test_pred_lr,
        y_test_prob_lr,
        "Logistic Regression"
    ),

    calculate_metrics(
        y_test,
        y_test_pred_dt,
        y_test_prob_dt,
        "Decision Tree"
    ),

    calculate_metrics(
        y_test,
        y_test_pred_rf,
        y_test_prob_rf,
        "Random Forest"
    ),

    calculate_metrics(
        y_test,
        y_test_pred_xgb,
        y_test_prob_xgb,
        "XGBoost"
    ),

    calculate_metrics(
        y_test,
        y_test_pred_dnn,
        y_test_prob_dnn,
        "Standard DNN"
    ),

    calculate_metrics(
        y_test,
        y_test_pred_lann,
        y_test_prob_lann,
        "LANN"
    )
])

print(
    test_results
    .round(4)
    .to_string(index=False)
)

In [ ]:
test_predictions = {
    "Logistic Regression": y_test_pred_lr,
    "Decision Tree": y_test_pred_dt,
    "Random Forest": y_test_pred_rf,
    "XGBoost": y_test_pred_xgb,
    "Standard DNN": y_test_pred_dnn,
    "LANN": y_test_pred_lann
}

for model_name, predictions in test_predictions.items():

    cm = confusion_matrix(
        y_test,
        predictions
    )

    print("\n" + "=" * 50)
    print(model_name)
    print("=" * 50)

    print(cm)

In [ ]:
confusion_summary = []

for model_name, predictions in test_predictions.items():

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        predictions
    ).ravel()

    confusion_summary.append({
        "Model": model_name,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

confusion_summary_df = pd.DataFrame(
    confusion_summary
)

print(
    confusion_summary_df.to_string(index=False)
)

In [ ]:
test_probabilities = {
    "Logistic Regression": y_test_prob_lr,
    "Decision Tree": y_test_prob_dt,
    "Random Forest": y_test_prob_rf,
    "XGBoost": y_test_prob_xgb,
    "Standard DNN": y_test_prob_dnn,
    "LANN": y_test_prob_lann
}

plt.figure(figsize=(9, 7))

for model_name, probabilities in test_probabilities.items():

    fpr, tpr, _ = roc_curve(
        y_test,
        probabilities
    )

    auc_score = roc_auc_score(
        y_test,
        probabilities
    )

    plt.plot(
        fpr,
        tpr,
        label=f"{model_name} (AUC = {auc_score:.3f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Final ROC Curves - Diabetes Classification")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.show()

In [ ]:
validation_results = all_validation_results.copy()

test_results.to_csv(
    "diabetes_test_results.csv",
    index=False
)

validation_results.to_csv(
    "diabetes_validation_results.csv",
    index=False
)

confusion_summary_df.to_csv(
    "diabetes_confusion_matrices.csv",
    index=False
)

attention_df.to_csv(
    "diabetes_lann_attention.csv",
    index=False
)

print("All diabetes results saved successfully.")

In [ ]:
comparison_table = validation_results.merge(
    test_results,
    on="Model",
    suffixes=("_Validation", "_Test")
)

print(
    comparison_table
    .round(4)
    .to_string(index=False)
)

In [ ]:
pcos_df = df[df["pcos_target"].notna()].copy()

print("PCOS shape:", pcos_df.shape)
print("\nTarget distribution:")
print(pcos_df["pcos_target"].value_counts())

print("\nSource datasets:")
print(pcos_df["source_dataset"].value_counts())

print("\nMissing values:")
print(
    pcos_df[
        [
            "age",
            "bmi",
            "height_cm",
            "weight_kg",
            "systolic_bp",
            "diastolic_bp",
            "waist_hip_ratio",
            "pulse_rate_bpm",
            "regular_exercise",
            "fast_food",
            "hip_circumference_cm",
            "waist_circumference_cm",
            "sleep_hours",
            "physical_activity_level",
            "smoking",
            "dietary_sugar_intake"
        ]
    ].isna().sum()
)

In [ ]:
pcos_numeric_features = [
    "age",
    "bmi",
    "height_cm",
    "weight_kg",
    "systolic_bp",
    "diastolic_bp",
    "waist_hip_ratio",
    "pulse_rate_bpm",
    "hip_circumference_cm",
    "waist_circumference_cm",
    "sleep_hours"
]

print(pcos_df[pcos_numeric_features].describe().T)

In [ ]:
print("Unique values in lifestyle variables:\n")

for col in [
    "regular_exercise",
    "fast_food",
    "physical_activity_level",
    "smoking",
    "dietary_sugar_intake"
]:
    print("\n", col)
    print(pcos_df[col].value_counts(dropna=False))

In [ ]:
pcos_features = [
    "age",
    "bmi",
    "height_cm",
    "weight_kg",
    "waist_hip_ratio",
    "systolic_bp",
    "diastolic_bp"
]

In [ ]:
pcos_df["patient_key"] = (
    pcos_df["source_dataset"].astype(str)
    + "_"
    + pcos_df["source_patient_id"].astype(str)
)

print("Total rows:", len(pcos_df))
print("Unique patient keys:", pcos_df["patient_key"].nunique())
print(
    "Duplicate patient keys:",
    pcos_df["patient_key"].duplicated().sum()
)

In [ ]:

pcos_model_df = pcos_df.copy()

pcos_model_df.loc[
    (pcos_model_df["systolic_bp"] < 70) |
    (pcos_model_df["systolic_bp"] > 250),
    "systolic_bp"
] = None

pcos_model_df.loc[
    (pcos_model_df["diastolic_bp"] < 40) |
    (pcos_model_df["diastolic_bp"] > 150),
    "diastolic_bp"
] = None

print("Missing values after cleaning:\n")

print(
    pcos_model_df[
        [
            "age",
            "bmi",
            "height_cm",
            "weight_kg",
            "waist_hip_ratio",
            "systolic_bp",
            "diastolic_bp"
        ]
    ].isna().sum()
)

In [ ]:
pcos_model_df["patient_key"] = (
    pcos_model_df["source_dataset"].astype(str)
    + "_"
    + pcos_model_df["source_patient_id"].astype(str)
)

print("Total rows:", len(pcos_model_df))
print("Unique patient keys:", pcos_model_df["patient_key"].nunique())
print(
    "Duplicate patient keys:",
    pcos_model_df["patient_key"].duplicated().sum()
)

In [ ]:
pcos_features = [
    "age",
    "bmi",
    "height_cm",
    "weight_kg",
    "waist_hip_ratio",
    "systolic_bp",
    "diastolic_bp"
]

X_pcos = pcos_model_df[pcos_features].copy()
y_pcos = pcos_model_df["pcos_target"].astype(int)

print("X shape:", X_pcos.shape)
print("y shape:", y_pcos.shape)
print("\nTarget distribution:")
print(y_pcos.value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

X_train_pcos, X_temp_pcos, y_train_pcos, y_temp_pcos = train_test_split(
    X_pcos,
    y_pcos,
    test_size=0.30,
    stratify=y_pcos,
    random_state=42
)

X_val_pcos, X_test_pcos, y_val_pcos, y_test_pcos = train_test_split(
    X_temp_pcos,
    y_temp_pcos,
    test_size=0.50,
    stratify=y_temp_pcos,
    random_state=42
)

print("Train:", X_train_pcos.shape, y_train_pcos.shape)
print("Validation:", X_val_pcos.shape, y_val_pcos.shape)
print("Test:", X_test_pcos.shape, y_test_pcos.shape)

print("\nTrain target:")
print(y_train_pcos.value_counts())

print("\nValidation target:")
print(y_val_pcos.value_counts())

print("\nTest target:")
print(y_test_pcos.value_counts())

In [ ]:
train_keys = set(
    pcos_model_df.loc[X_train_pcos.index, "patient_key"]
)

val_keys = set(
    pcos_model_df.loc[X_val_pcos.index, "patient_key"]
)

test_keys = set(
    pcos_model_df.loc[X_test_pcos.index, "patient_key"]
)

print("Train ∩ Validation:", len(train_keys & val_keys))
print("Train ∩ Test:", len(train_keys & test_keys))
print("Validation ∩ Test:", len(val_keys & test_keys))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

pcos_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

X_train_pcos_processed = pcos_preprocessor.fit_transform(X_train_pcos)
X_val_pcos_processed = pcos_preprocessor.transform(X_val_pcos)
X_test_pcos_processed = pcos_preprocessor.transform(X_test_pcos)

print("Processed train shape:", X_train_pcos_processed.shape)
print("Processed validation shape:", X_val_pcos_processed.shape)
print("Processed test shape:", X_test_pcos_processed.shape)

In [ ]:
pcos_feature_names = pcos_features

print("Features:")
for i, feature in enumerate(pcos_feature_names):
    print(i, feature)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

lr_pcos = LogisticRegression(
    max_iter=1000,
    random_state=42
)

dt_pcos = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

rf_pcos = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

xgb_pcos = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

lr_pcos.fit(
    X_train_pcos_processed,
    y_train_pcos
)

dt_pcos.fit(
    X_train_pcos_processed,
    y_train_pcos
)

rf_pcos.fit(
    X_train_pcos_processed,
    y_train_pcos
)

xgb_pcos.fit(
    X_train_pcos_processed,
    y_train_pcos
)

print("All four PCOS models trained successfully.")

In [ ]:
y_val_pred_lr_pcos = lr_pcos.predict(X_val_pcos_processed)
y_val_prob_lr_pcos = lr_pcos.predict_proba(X_val_pcos_processed)[:, 1]

y_val_pred_dt_pcos = dt_pcos.predict(X_val_pcos_processed)
y_val_prob_dt_pcos = dt_pcos.predict_proba(X_val_pcos_processed)[:, 1]

y_val_pred_rf_pcos = rf_pcos.predict(X_val_pcos_processed)
y_val_prob_rf_pcos = rf_pcos.predict_proba(X_val_pcos_processed)[:, 1]

y_val_pred_xgb_pcos = xgb_pcos.predict(X_val_pcos_processed)
y_val_prob_xgb_pcos = xgb_pcos.predict_proba(X_val_pcos_processed)[:, 1]

print("Validation predictions generated.")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def calculate_pcos_metrics(
    y_true,
    y_pred,
    y_prob,
    model_name
):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob)
    }

pcos_validation_results = pd.DataFrame([
    calculate_pcos_metrics(
        y_val_pcos,
        y_val_pred_lr_pcos,
        y_val_prob_lr_pcos,
        "Logistic Regression"
    ),
    calculate_pcos_metrics(
        y_val_pcos,
        y_val_pred_dt_pcos,
        y_val_prob_dt_pcos,
        "Decision Tree"
    ),
    calculate_pcos_metrics(
        y_val_pcos,
        y_val_pred_rf_pcos,
        y_val_prob_rf_pcos,
        "Random Forest"
    ),
    calculate_pcos_metrics(
        y_val_pcos,
        y_val_pred_xgb_pcos,
        y_val_prob_xgb_pcos,
        "XGBoost"
    )
])

print(pcos_validation_results.round(4).to_string(index=False))

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)

dnn_pcos = Sequential([
    Input(shape=(X_train_pcos_processed.shape[1],)),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

dnn_pcos.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping_dnn_pcos = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history_dnn_pcos = dnn_pcos.fit(
    X_train_pcos_processed,
    y_train_pcos,
    validation_data=(
        X_val_pcos_processed,
        y_val_pcos
    ),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping_dnn_pcos],
    verbose=1
)

In [ ]:
y_val_prob_dnn_pcos = dnn_pcos.predict(
    X_val_pcos_processed,
    verbose=0
).ravel()

y_val_pred_dnn_pcos = (
    y_val_prob_dnn_pcos >= 0.5
).astype(int)

dnn_pcos_results = calculate_pcos_metrics(
    y_val_pcos,
    y_val_pred_dnn_pcos,
    y_val_prob_dnn_pcos,
    "Standard DNN"
)

print(pd.DataFrame([dnn_pcos_results]).round(4).to_string(index=False))

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, Multiply
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

tf.random.set_seed(42)

input_dim_pcos = X_train_pcos_processed.shape[1]

inputs_pcos = Input(shape=(input_dim_pcos,))

attention_scores_pcos = Dense(
    input_dim_pcos,
    activation="softmax",
    name="feature_attention"
)(inputs_pcos)

attended_features_pcos = Multiply()([
    inputs_pcos,
    attention_scores_pcos
])

x_pcos = Dense(64, activation="relu")(attended_features_pcos)
x_pcos = Dropout(0.2)(x_pcos)

x_pcos = Dense(32, activation="relu")(x_pcos)
x_pcos = Dropout(0.2)(x_pcos)

outputs_pcos = Dense(1, activation="sigmoid")(x_pcos)

lann_pcos = Model(
    inputs=inputs_pcos,
    outputs=outputs_pcos,
    name="PCOS_Lifestyle_Attention_Neural_Network"
)

lann_pcos.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping_lann_pcos = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history_lann_pcos = lann_pcos.fit(
    X_train_pcos_processed,
    y_train_pcos,
    validation_data=(
        X_val_pcos_processed,
        y_val_pcos
    ),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping_lann_pcos],
    verbose=1
)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

y_val_prob_lann_pcos = lann_pcos.predict(
    X_val_pcos_processed,
    verbose=0
).ravel()

y_val_pred_lann_pcos = (
    y_val_prob_lann_pcos >= 0.5
).astype(int)

lann_pcos_results = {
    "Model": "LANN",
    "Accuracy": accuracy_score(
        y_val_pcos,
        y_val_pred_lann_pcos
    ),
    "Precision": precision_score(
        y_val_pcos,
        y_val_pred_lann_pcos,
        zero_division=0
    ),
    "Recall": recall_score(
        y_val_pcos,
        y_val_pred_lann_pcos,
        zero_division=0
    ),
    "F1": f1_score(
        y_val_pcos,
        y_val_pred_lann_pcos,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_val_pcos,
        y_val_prob_lann_pcos
    )
}

print(
    pd.DataFrame([lann_pcos_results])
    .round(4)
    .to_string(index=False)
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

pcos_test_results = []

pcos_models = {
    "Logistic Regression": lr_pcos,
    "Decision Tree": dt_pcos,
    "Random Forest": rf_pcos,
    "XGBoost": xgb_pcos
}

for model_name, model in pcos_models.items():

    y_prob = model.predict_proba(
        X_test_pcos_processed
    )[:, 1]

    y_pred = (
        y_prob >= 0.5
    ).astype(int)

    pcos_test_results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(
            y_test_pcos,
            y_pred
        ),
        "Precision": precision_score(
            y_test_pcos,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test_pcos,
            y_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_test_pcos,
            y_pred,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_test_pcos,
            y_prob
        )
    })

y_prob_dnn = dnn_pcos.predict(
    X_test_pcos_processed,
    verbose=0
).ravel()

y_pred_dnn = (
    y_prob_dnn >= 0.5
).astype(int)

pcos_test_results.append({
    "Model": "Standard DNN",
    "Accuracy": accuracy_score(
        y_test_pcos,
        y_pred_dnn
    ),
    "Precision": precision_score(
        y_test_pcos,
        y_pred_dnn,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test_pcos,
        y_pred_dnn,
        zero_division=0
    ),
    "F1": f1_score(
        y_test_pcos,
        y_pred_dnn,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_test_pcos,
        y_prob_dnn
    )
})

y_prob_lann = lann_pcos.predict(
    X_test_pcos_processed,
    verbose=0
).ravel()

y_pred_lann = (
    y_prob_lann >= 0.5
).astype(int)

pcos_test_results.append({
    "Model": "LANN",
    "Accuracy": accuracy_score(
        y_test_pcos,
        y_pred_lann
    ),
    "Precision": precision_score(
        y_test_pcos,
        y_pred_lann,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test_pcos,
        y_pred_lann,
        zero_division=0
    ),
    "F1": f1_score(
        y_test_pcos,
        y_pred_lann,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_test_pcos,
        y_prob_lann
    )
})

pcos_test_results_df = pd.DataFrame(
    pcos_test_results
)

print(
    pcos_test_results_df
    .round(4)
    .to_string(index=False)
)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

cm_lann_pcos = confusion_matrix(
    y_test_pcos,
    y_pred_lann
)

print("LANN Confusion Matrix:")
print(cm_lann_pcos)

print("\nLANN Classification Report:")
print(
    classification_report(
        y_test_pcos,
        y_pred_lann,
        target_names=["No PCOS", "PCOS"],
        zero_division=0
    )
)

In [ ]:
print("LANN Validation")
print("Accuracy :", round(lann_pcos_results["Accuracy"], 4))
print("Precision:", round(lann_pcos_results["Precision"], 4))
print("Recall   :", round(lann_pcos_results["Recall"], 4))
print("F1       :", round(lann_pcos_results["F1"], 4))
print("ROC-AUC  :", round(lann_pcos_results["ROC-AUC"], 4))

print("\nLANN Test")
print("Accuracy :", round(accuracy_score(y_test_pcos, y_pred_lann), 4))
print("Precision:", round(precision_score(y_test_pcos, y_pred_lann, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test_pcos, y_pred_lann, zero_division=0), 4))
print("F1       :", round(f1_score(y_test_pcos, y_pred_lann, zero_division=0), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test_pcos, y_prob_lann), 4))

In [ ]:
from sklearn.metrics import confusion_matrix

cm_lann_pcos = confusion_matrix(
    y_test_pcos,
    y_pred_lann
)

print("LANN Confusion Matrix:")
print(cm_lann_pcos)

tn, fp, fn, tp = cm_lann_pcos.ravel()

print("\nTN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay(
    confusion_matrix=cm_lann_pcos,
    display_labels=["No PCOS", "PCOS"]
).plot()

plt.title("PCOS LANN - Test Confusion Matrix")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

pcos_test_predictions = {}

y_prob_lr_pcos = lr_pcos.predict_proba(
    X_test_pcos_processed
)[:, 1]

y_pred_lr_pcos = (
    y_prob_lr_pcos >= 0.5
).astype(int)

pcos_test_predictions["Logistic Regression"] = (
    y_test_pcos,
    y_pred_lr_pcos
)

y_prob_dt_pcos = dt_pcos.predict_proba(
    X_test_pcos_processed
)[:, 1]

y_pred_dt_pcos = (
    y_prob_dt_pcos >= 0.5
).astype(int)

pcos_test_predictions["Decision Tree"] = (
    y_test_pcos,
    y_pred_dt_pcos
)

y_prob_rf_pcos = rf_pcos.predict_proba(
    X_test_pcos_processed
)[:, 1]

y_pred_rf_pcos = (
    y_prob_rf_pcos >= 0.5
).astype(int)

pcos_test_predictions["Random Forest"] = (
    y_test_pcos,
    y_pred_rf_pcos
)

y_prob_xgb_pcos = xgb_pcos.predict_proba(
    X_test_pcos_processed
)[:, 1]

y_pred_xgb_pcos = (
    y_prob_xgb_pcos >= 0.5
).astype(int)

pcos_test_predictions["XGBoost"] = (
    y_test_pcos,
    y_pred_xgb_pcos
)

y_prob_dnn_pcos_test = dnn_pcos.predict(
    X_test_pcos_processed,
    verbose=0
).ravel()

y_pred_dnn_pcos_test = (
    y_prob_dnn_pcos_test >= 0.5
).astype(int)

pcos_test_predictions["Standard DNN"] = (
    y_test_pcos,
    y_pred_dnn_pcos_test
)

y_prob_lann_pcos_test = lann_pcos.predict(
    X_test_pcos_processed,
    verbose=0
).ravel()

y_pred_lann_pcos_test = (
    y_prob_lann_pcos_test >= 0.5
).astype(int)

pcos_test_predictions["LANN"] = (
    y_test_pcos,
    y_pred_lann_pcos_test
)

for model_name, (y_true, y_pred) in pcos_test_predictions.items():

    cm = confusion_matrix(y_true, y_pred)

    print("\n" + model_name)
    print(cm)

    tn, fp, fn, tp = cm.ravel()

    print(
        f"TN={tn}, FP={fp}, FN={fn}, TP={tp}"
    )

In [ ]:
for model_name, (y_true, y_pred) in pcos_test_predictions.items():

    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(5, 4))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["No PCOS", "PCOS"]
    )

    disp.plot(
        ax=ax,
        values_format="d"
    )

    ax.set_title(
        f"PCOS - {model_name} - Test Confusion Matrix"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

pcos_probabilities = {
    "Logistic Regression": y_prob_lr_pcos,
    "Decision Tree": y_prob_dt_pcos,
    "Random Forest": y_prob_rf_pcos,
    "XGBoost": y_prob_xgb_pcos,
    "Standard DNN": y_prob_dnn_pcos_test,
    "LANN": y_prob_lann_pcos_test
}

plt.figure(figsize=(9, 7))

for model_name, y_prob in pcos_probabilities.items():

    fpr, tpr, _ = roc_curve(
        y_test_pcos,
        y_prob
    )

    auc_value = roc_auc_score(
        y_test_pcos,
        y_prob
    )

    plt.plot(
        fpr,
        tpr,
        label=f"{model_name} (AUC = {auc_value:.3f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("PCOS Model Comparison - ROC Curves")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

attention_layer_pcos = lann_pcos.get_layer(
    "feature_attention"
)

attention_weights_pcos = attention_layer_pcos.get_weights()

attention_kernel_pcos = attention_weights_pcos[0]
attention_bias_pcos = attention_weights_pcos[1]

X_attention_pcos = X_val_pcos_processed

attention_logits_pcos = (
    np.dot(
        X_attention_pcos,
        attention_kernel_pcos
    )
    + attention_bias_pcos
)

attention_scores_pcos = np.exp(
    attention_logits_pcos
)

attention_scores_pcos = (
    attention_scores_pcos /
    attention_scores_pcos.sum(
        axis=1,
        keepdims=True
    )
)

mean_attention_pcos = (
    attention_scores_pcos.mean(axis=0)
)

attention_df_pcos = pd.DataFrame({
    "Feature": pcos_feature_names,
    "Mean Attention": mean_attention_pcos
})

attention_df_pcos = attention_df_pcos.sort_values(
    "Mean Attention",
    ascending=False
)

print(
    attention_df_pcos
    .round(6)
    .to_string(index=False)
)

In [ ]:
plt.figure(figsize=(9, 5))

plt.barh(
    attention_df_pcos["Feature"],
    attention_df_pcos["Mean Attention"]
)

plt.xlabel("Mean Attention Weight")
plt.ylabel("Feature")
plt.title("PCOS LANN Feature Attention")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
!pip install -q shap
import shap

explainer_lr_pcos = shap.LinearExplainer(
    lr_pcos,
    X_train_pcos_processed
)

shap_values_lr_pcos = explainer_lr_pcos(
    X_val_pcos_processed
)

In [ ]:
shap.summary_plot(
    shap_values_lr_pcos.values,
    X_val_pcos_processed,
    feature_names=pcos_feature_names,
    show=False
)

plt.title(
    "PCOS Logistic Regression - SHAP Feature Importance"
)

plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(
    shap_values_lr_pcos.values,
    X_val_pcos_processed,
    feature_names=pcos_feature_names,
    plot_type="bar",
    show=False
)

plt.title(
    "PCOS Logistic Regression - Mean Absolute SHAP Values"
)

plt.tight_layout()
plt.show()

In [ ]:
background_size_pcos = min(
    100,
    len(X_train_pcos_processed)
)

background_pcos = (
    X_train_pcos_processed[
        :background_size_pcos
    ]
)

explainer_lann_pcos = shap.Explainer(
    lann_pcos,
    background_pcos
)

shap_values_lann_pcos = explainer_lann_pcos(
    X_val_pcos_processed
)

In [ ]:
shap.summary_plot(
    shap_values_lann_pcos.values,
    X_val_pcos_processed,
    feature_names=pcos_feature_names,
    show=False
)

plt.title(
    "PCOS LANN - SHAP Feature Importance"
)

plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(
    shap_values_lann_pcos.values,
    X_val_pcos_processed,
    feature_names=pcos_feature_names,
    plot_type="bar",
    show=False
)

plt.title(
    "PCOS LANN - Mean Absolute SHAP Values"
)

plt.tight_layout()
plt.show()

In [ ]:
pcos_test_results_df.round(4)
pcos_report_results = pcos_test_results_df.copy()

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC"
]

pcos_report_results[metric_columns] = (
    pcos_report_results[metric_columns] * 100
).round(2)

print(
    pcos_report_results.to_string(
        index=False
    )
)

In [ ]:
pcos_lifestyle_features = [
    "regular_exercise",
    "fast_food",
    "physical_activity_level",
    "smoking",
    "dietary_sugar_intake",
    "sleep_hours"
]

pcos_lifestyle_df = pcos_model_df[
    pcos_lifestyle_features + ["pcos_target", "patient_key"]
].copy()

print("Shape:", pcos_lifestyle_df.shape)

print("\nMissing values:")
print(
    pcos_lifestyle_df[
        pcos_lifestyle_features
    ].isnull().sum()
)

print("\nTarget distribution:")
print(
    pcos_lifestyle_df["pcos_target"].value_counts()
)

In [ ]:
for feature in pcos_lifestyle_features:

    print("\n" + feature)

    print(
        pcos_lifestyle_df[feature]
        .value_counts(dropna=False)
    )

In [ ]:
from sklearn.model_selection import train_test_split

X_pcos_lifestyle = pcos_lifestyle_df[
    pcos_lifestyle_features
].copy()

y_pcos_lifestyle = pcos_lifestyle_df[
    "pcos_target"
].astype(int)

X_train_lifestyle, X_temp_lifestyle, \
y_train_lifestyle, y_temp_lifestyle = train_test_split(
    X_pcos_lifestyle,
    y_pcos_lifestyle,
    test_size=0.30,
    stratify=y_pcos_lifestyle,
    random_state=42
)

X_val_lifestyle, X_test_lifestyle, \
y_val_lifestyle, y_test_lifestyle = train_test_split(
    X_temp_lifestyle,
    y_temp_lifestyle,
    test_size=0.50,
    stratify=y_temp_lifestyle,
    random_state=42
)

print("Training:", X_train_lifestyle.shape)
print("Validation:", X_val_lifestyle.shape)
print("Test:", X_test_lifestyle.shape)

print("\nTraining target:")
print(y_train_lifestyle.value_counts())

print("\nValidation target:")
print(y_val_lifestyle.value_counts())

print("\nTest target:")
print(y_test_lifestyle.value_counts())

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

lifestyle_preprocessor = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

X_train_lifestyle_processed = (
    lifestyle_preprocessor.fit_transform(
        X_train_lifestyle
    )
)

X_val_lifestyle_processed = (
    lifestyle_preprocessor.transform(
        X_val_lifestyle
    )
)

X_test_lifestyle_processed = (
    lifestyle_preprocessor.transform(
        X_test_lifestyle
    )
)

print(
    "Train:",
    X_train_lifestyle_processed.shape
)

print(
    "Validation:",
    X_val_lifestyle_processed.shape
)

print(
    "Test:",
    X_test_lifestyle_processed.shape
)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

lr_lifestyle = LogisticRegression(
    max_iter=1000,
    random_state=42
)

dt_lifestyle = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

rf_lifestyle = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

xgb_lifestyle = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

traditional_lifestyle_models = {
    "Logistic Regression": lr_lifestyle,
    "Decision Tree": dt_lifestyle,
    "Random Forest": rf_lifestyle,
    "XGBoost": xgb_lifestyle
}

for name, model in traditional_lifestyle_models.items():

    model.fit(
        X_train_lifestyle_processed,
        y_train_lifestyle
    )

    print(name, "trained")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

lifestyle_validation_results = []

for name, model in traditional_lifestyle_models.items():

    y_prob = model.predict_proba(
        X_val_lifestyle_processed
    )[:, 1]

    y_pred = (
        y_prob >= 0.5
    ).astype(int)

    lifestyle_validation_results.append({
        "Model": name,
        "Accuracy": accuracy_score(
            y_val_lifestyle,
            y_pred
        ),
        "Precision": precision_score(
            y_val_lifestyle,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_val_lifestyle,
            y_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_val_lifestyle,
            y_pred,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_val_lifestyle,
            y_prob
        )
    })

lifestyle_validation_df = pd.DataFrame(
    lifestyle_validation_results
)

print(
    lifestyle_validation_df
    .round(4)
    .to_string(index=False)
)

In [ ]:
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)

dnn_lifestyle = Sequential([
    Input(
        shape=(
            X_train_lifestyle_processed.shape[1],
        )
    ),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

dnn_lifestyle.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping_dnn_lifestyle = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history_dnn_lifestyle = dnn_lifestyle.fit(
    X_train_lifestyle_processed,
    y_train_lifestyle,
    validation_data=(
        X_val_lifestyle_processed,
        y_val_lifestyle
    ),
    epochs=100,
    batch_size=32,
    callbacks=[
        early_stopping_dnn_lifestyle
    ],
    verbose=1
)

In [ ]:
y_val_prob_dnn_lifestyle = dnn_lifestyle.predict(
    X_val_lifestyle_processed,
    verbose=0
).ravel()

y_val_pred_dnn_lifestyle = (
    y_val_prob_dnn_lifestyle >= 0.5
).astype(int)

dnn_lifestyle_result = {
    "Model": "Standard DNN",
    "Accuracy": accuracy_score(
        y_val_lifestyle,
        y_val_pred_dnn_lifestyle
    ),
    "Precision": precision_score(
        y_val_lifestyle,
        y_val_pred_dnn_lifestyle,
        zero_division=0
    ),
    "Recall": recall_score(
        y_val_lifestyle,
        y_val_pred_dnn_lifestyle,
        zero_division=0
    ),
    "F1": f1_score(
        y_val_lifestyle,
        y_val_pred_dnn_lifestyle,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_val_lifestyle,
        y_val_prob_dnn_lifestyle
    )
}

print(
    pd.DataFrame([dnn_lifestyle_result])
    .round(4)
    .to_string(index=False)
)

In [ ]:
from tensorflow.keras.layers import Multiply

tf.random.set_seed(42)

input_dim_lifestyle = (
    X_train_lifestyle_processed.shape[1]
)

inputs_lifestyle = Input(
    shape=(input_dim_lifestyle,)
)

attention_scores_lifestyle = Dense(
    input_dim_lifestyle,
    activation="softmax",
    name="feature_attention_lifestyle"
)(inputs_lifestyle)

attended_features_lifestyle = Multiply()([
    inputs_lifestyle,
    attention_scores_lifestyle
])

x_lifestyle = Dense(
    64,
    activation="relu"
)(attended_features_lifestyle)

x_lifestyle = Dropout(0.2)(x_lifestyle)

x_lifestyle = Dense(
    32,
    activation="relu"
)(x_lifestyle)

x_lifestyle = Dropout(0.2)(x_lifestyle)

outputs_lifestyle = Dense(
    1,
    activation="sigmoid"
)(x_lifestyle)

lann_lifestyle = Model(
    inputs=inputs_lifestyle,
    outputs=outputs_lifestyle,
    name="Lifestyle_Attention_Neural_Network"
)

lann_lifestyle.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping_lann_lifestyle = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history_lann_lifestyle = lann_lifestyle.fit(
    X_train_lifestyle_processed,
    y_train_lifestyle,
    validation_data=(
        X_val_lifestyle_processed,
        y_val_lifestyle
    ),
    epochs=100,
    batch_size=32,
    callbacks=[
        early_stopping_lann_lifestyle
    ],
    verbose=1
)

In [ ]:
y_val_prob_lann_lifestyle = lann_lifestyle.predict(
    X_val_lifestyle_processed,
    verbose=0
).ravel()

y_val_pred_lann_lifestyle = (
    y_val_prob_lann_lifestyle >= 0.5
).astype(int)

lann_lifestyle_result = {
    "Model": "LANN",
    "Accuracy": accuracy_score(
        y_val_lifestyle,
        y_val_pred_lann_lifestyle
    ),
    "Precision": precision_score(
        y_val_lifestyle,
        y_val_pred_lann_lifestyle,
        zero_division=0
    ),
    "Recall": recall_score(
        y_val_lifestyle,
        y_val_pred_lann_lifestyle,
        zero_division=0
    ),
    "F1": f1_score(
        y_val_lifestyle,
        y_val_pred_lann_lifestyle,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_val_lifestyle,
        y_val_prob_lann_lifestyle
    )
}

print(
    pd.DataFrame([lann_lifestyle_result])
    .round(4)
    .to_string(index=False)
)

In [ ]:
lifestyle_validation_results.append(
    dnn_lifestyle_result
)

lifestyle_validation_results.append(
    lann_lifestyle_result
)

lifestyle_validation_final = pd.DataFrame(
    lifestyle_validation_results
)

print(
    lifestyle_validation_final
    .round(4)
    .to_string(index=False)
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

lifestyle_test_results = []

lifestyle_test_predictions = {}

# Traditional ML models
for model_name, model in traditional_lifestyle_models.items():

    y_prob = model.predict_proba(
        X_test_lifestyle_processed
    )[:, 1]

    y_pred = (
        y_prob >= 0.5
    ).astype(int)

    lifestyle_test_results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(
            y_test_lifestyle,
            y_pred
        ),
        "Precision": precision_score(
            y_test_lifestyle,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test_lifestyle,
            y_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_test_lifestyle,
            y_pred,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_test_lifestyle,
            y_prob
        )
    })

    lifestyle_test_predictions[model_name] = (
        y_test_lifestyle,
        y_pred,
        y_prob
    )


# Standard DNN
y_prob_dnn_lifestyle_test = dnn_lifestyle.predict(
    X_test_lifestyle_processed,
    verbose=0
).ravel()

y_pred_dnn_lifestyle_test = (
    y_prob_dnn_lifestyle_test >= 0.5
).astype(int)

lifestyle_test_results.append({
    "Model": "Standard DNN",
    "Accuracy": accuracy_score(
        y_test_lifestyle,
        y_pred_dnn_lifestyle_test
    ),
    "Precision": precision_score(
        y_test_lifestyle,
        y_pred_dnn_lifestyle_test,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test_lifestyle,
        y_pred_dnn_lifestyle_test,
        zero_division=0
    ),
    "F1": f1_score(
        y_test_lifestyle,
        y_pred_dnn_lifestyle_test,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_test_lifestyle,
        y_prob_dnn_lifestyle_test
    )
})

lifestyle_test_predictions["Standard DNN"] = (
    y_test_lifestyle,
    y_pred_dnn_lifestyle_test,
    y_prob_dnn_lifestyle_test
)


# LANN
y_prob_lann_lifestyle_test = lann_lifestyle.predict(
    X_test_lifestyle_processed,
    verbose=0
).ravel()

y_pred_lann_lifestyle_test = (
    y_prob_lann_lifestyle_test >= 0.5
).astype(int)

lifestyle_test_results.append({
    "Model": "LANN",
    "Accuracy": accuracy_score(
        y_test_lifestyle,
        y_pred_lann_lifestyle_test
    ),
    "Precision": precision_score(
        y_test_lifestyle,
        y_pred_lann_lifestyle_test,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test_lifestyle,
        y_pred_lann_lifestyle_test,
        zero_division=0
    ),
    "F1": f1_score(
        y_test_lifestyle,
        y_pred_lann_lifestyle_test,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_test_lifestyle,
        y_prob_lann_lifestyle_test
    )
})

lifestyle_test_predictions["LANN"] = (
    y_test_lifestyle,
    y_pred_lann_lifestyle_test,
    y_prob_lann_lifestyle_test
)


lifestyle_test_df = pd.DataFrame(
    lifestyle_test_results
)

print(
    lifestyle_test_df
    .round(4)
    .to_string(index=False)
)

In [ ]:
from sklearn.metrics import confusion_matrix

print("PCOS Lifestyle - Test Confusion Matrices")

for model_name, (
    y_true,
    y_pred,
    y_prob
) in lifestyle_test_predictions.items():

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    tn, fp, fn, tp = cm.ravel()

    print("\n" + model_name)
    print(cm)
    print(
        f"TN={tn}, FP={fp}, FN={fn}, TP={tp}"
    )

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

for model_name, (
    y_true,
    y_pred,
    y_prob
) in lifestyle_test_predictions.items():

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    fig, ax = plt.subplots(
        figsize=(5, 4)
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[
            "No PCOS",
            "PCOS"
        ]
    )

    disp.plot(
        ax=ax,
        values_format="d"
    )

    ax.set_title(
        f"PCOS Lifestyle - {model_name}"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 7))

for model_name, (
    y_true,
    y_pred,
    y_prob
) in lifestyle_test_predictions.items():

    fpr, tpr, thresholds = roc_curve(
        y_true,
        y_prob
    )

    auc_value = roc_auc_score(
        y_true,
        y_prob
    )

    plt.plot(
        fpr,
        tpr,
        label=f"{model_name} (AUC = {auc_value:.3f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "PCOS Lifestyle Model Comparison - ROC Curves"
)

plt.legend()

plt.grid()

plt.tight_layout()

plt.show()

In [ ]:
import numpy as np
import pandas as pd

attention_layer_lifestyle = (
    lann_lifestyle.get_layer(
        "feature_attention_lifestyle"
    )
)

attention_weights_lifestyle = (
    attention_layer_lifestyle.get_weights()
)

attention_kernel_lifestyle = (
    attention_weights_lifestyle[0]
)

attention_bias_lifestyle = (
    attention_weights_lifestyle[1]
)

attention_logits_lifestyle = (
    np.dot(
        X_val_lifestyle_processed,
        attention_kernel_lifestyle
    )
    + attention_bias_lifestyle
)

attention_scores_lifestyle = np.exp(
    attention_logits_lifestyle
)

attention_scores_lifestyle = (
    attention_scores_lifestyle /
    attention_scores_lifestyle.sum(
        axis=1,
        keepdims=True
    )
)

mean_attention_lifestyle = (
    attention_scores_lifestyle.mean(
        axis=0
    )
)

attention_lifestyle_df = pd.DataFrame({
    "Feature": pcos_lifestyle_features,
    "Mean Attention": mean_attention_lifestyle
})

attention_lifestyle_df = (
    attention_lifestyle_df
    .sort_values(
        "Mean Attention",
        ascending=False
    )
)

print(
    attention_lifestyle_df
    .round(6)
    .to_string(index=False)
)

In [ ]:
plt.figure(figsize=(9, 5))

plt.barh(
    attention_lifestyle_df["Feature"],
    attention_lifestyle_df["Mean Attention"]
)

plt.xlabel(
    "Mean Attention Weight"
)

plt.ylabel(
    "Lifestyle Feature"
)

plt.title(
    "PCOS Lifestyle LANN - Feature Attention"
)

plt.gca().invert_yaxis()

plt.tight_layout()

plt.show()

In [ ]:
!pip install -q shap
import shap

explainer_lr_lifestyle = shap.LinearExplainer(
    lr_lifestyle,
    X_train_lifestyle_processed
)

shap_values_lr_lifestyle = (
    explainer_lr_lifestyle(
        X_val_lifestyle_processed
    )
)

In [ ]:
shap.summary_plot(
    shap_values_lr_lifestyle.values,
    X_val_lifestyle_processed,
    feature_names=pcos_lifestyle_features,
    show=False
)

plt.title(
    "PCOS Lifestyle - Logistic Regression SHAP"
)

plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(
    shap_values_lr_lifestyle.values,
    X_val_lifestyle_processed,
    feature_names=pcos_lifestyle_features,
    plot_type="bar",
    show=False
)

plt.title(
    "PCOS Lifestyle - SHAP Feature Importance"
)

plt.tight_layout()
plt.show()

In [ ]:
background_size_lifestyle = min(
    100,
    len(X_train_lifestyle_processed)
)

background_lifestyle = (
    X_train_lifestyle_processed[
        :background_size_lifestyle
    ]
)

explainer_lann_lifestyle = shap.Explainer(
    lann_lifestyle,
    background_lifestyle
)

shap_values_lann_lifestyle = (
    explainer_lann_lifestyle(
        X_val_lifestyle_processed
    )
)

In [ ]:
shap.summary_plot(
    shap_values_lann_lifestyle.values,
    X_val_lifestyle_processed,
    feature_names=pcos_lifestyle_features,
    show=False
)

plt.title(
    "PCOS Lifestyle LANN - SHAP Feature Importance"
)

plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(
    shap_values_lann_lifestyle.values,
    X_val_lifestyle_processed,
    feature_names=pcos_lifestyle_features,
    plot_type="bar",
    show=False
)

plt.title(
    "PCOS Lifestyle LANN - Mean Absolute SHAP Values"
)

plt.tight_layout()
plt.show()

In [ ]:
lifestyle_test_report = lifestyle_test_df.copy()

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC"
]

lifestyle_test_report[
    metric_columns
] = (
    lifestyle_test_report[
        metric_columns
    ] * 100
).round(2)

print(
    lifestyle_test_report.to_string(
        index=False
    )
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score
)

print("=" * 70)
print("PCOS LIFESTYLE EXPERIMENT - FINAL ANALYSIS")
print("=" * 70)

print("\n1. FINAL TEST RESULTS")
print("-" * 70)

print(
    lifestyle_test_df.to_string(index=False)
)

lifestyle_test_report = lifestyle_test_df.copy()

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC"
]

lifestyle_test_report[metric_columns] = (
    lifestyle_test_report[metric_columns] * 100
).round(2)

print("\nPercentage Format:")
print(
    lifestyle_test_report.to_string(index=False)
)

print("\n" + "=" * 70)
print("2. CONFUSION MATRICES")
print("=" * 70)

confusion_matrix_results = {}

for model_name, (
    y_true,
    y_pred,
    y_prob
) in lifestyle_test_predictions.items():

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    confusion_matrix_results[model_name] = cm

    tn, fp, fn, tp = cm.ravel()

    print("\n" + model_name)
    print("Confusion Matrix:")
    print(cm)

    print(
        f"TN = {tn}"
    )
    print(
        f"FP = {fp}"
    )
    print(
        f"FN = {fn}"
    )
    print(
        f"TP = {tp}"
    )

print("\n" + "=" * 70)
print("3. CONFUSION MATRIX PLOTS")
print("=" * 70)

for model_name, (
    y_true,
    y_pred,
    y_prob
) in lifestyle_test_predictions.items():

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    fig, ax = plt.subplots(
        figsize=(5, 4)
    )

    display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[
            "No PCOS",
            "PCOS"
        ]
    )

    display.plot(
        ax=ax,
        values_format="d"
    )

    ax.set_title(
        f"PCOS Lifestyle - {model_name}\nTest Confusion Matrix"
    )

    plt.tight_layout()
    plt.show()


print("\n" + "=" * 70)
print("4. ROC CURVE")
print("=" * 70)

plt.figure(
    figsize=(9, 7)
)

roc_results = []

for model_name, (
    y_true,
    y_pred,
    y_prob
) in lifestyle_test_predictions.items():

    fpr, tpr, thresholds = roc_curve(
        y_true,
        y_prob
    )

    auc_value = roc_auc_score(
        y_true,
        y_prob
    )

    roc_results.append({
        "Model": model_name,
        "ROC-AUC": auc_value
    })

    plt.plot(
        fpr,
        tpr,
        label=f"{model_name} (AUC = {auc_value:.3f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "PCOS Lifestyle Models - ROC Curves"
)

plt.legend()

plt.grid()

plt.tight_layout()

plt.show()

roc_results_df = pd.DataFrame(
    roc_results
)

print("\nROC-AUC values:")
print(
    roc_results_df
    .sort_values(
        "ROC-AUC",
        ascending=False
    )
    .to_string(index=False)
)

print("\n" + "=" * 70)
print("5. LANN FEATURE ATTENTION")
print("=" * 70)

attention_layer_lifestyle = (
    lann_lifestyle.get_layer(
        "feature_attention_lifestyle"
    )
)

attention_weights_lifestyle = (
    attention_layer_lifestyle.get_weights()
)

attention_kernel_lifestyle = (
    attention_weights_lifestyle[0]
)

attention_bias_lifestyle = (
    attention_weights_lifestyle[1]
)

attention_logits_lifestyle = (
    np.dot(
        X_val_lifestyle_processed,
        attention_kernel_lifestyle
    )
    + attention_bias_lifestyle
)

attention_scores_lifestyle = np.exp(
    attention_logits_lifestyle
)

attention_scores_lifestyle = (
    attention_scores_lifestyle
    / attention_scores_lifestyle.sum(
        axis=1,
        keepdims=True
    )
)

mean_attention_lifestyle = (
    attention_scores_lifestyle.mean(
        axis=0
    )
)

attention_lifestyle_df = pd.DataFrame({
    "Feature": pcos_lifestyle_features,
    "Mean Attention": mean_attention_lifestyle
})

attention_lifestyle_df = (
    attention_lifestyle_df
    .sort_values(
        "Mean Attention",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    attention_lifestyle_df.round(6)
    .to_string(index=False)
)

plt.figure(
    figsize=(9, 5)
)

plt.barh(
    attention_lifestyle_df["Feature"],
    attention_lifestyle_df["Mean Attention"]
)

plt.xlabel(
    "Mean Attention Weight"
)

plt.ylabel(
    "Lifestyle Feature"
)

plt.title(
    "PCOS Lifestyle LANN - Feature Attention"
)

plt.gca().invert_yaxis()

plt.tight_layout()

plt.show()

print("\n" + "=" * 70)
print("7. SHAP ANALYSIS")
print("=" * 70)

try:
    import shap
    print("SHAP is already installed.")
except ImportError:
    print("Installing SHAP...")
    !pip install -q shap
    import shap

print("\nRunning SHAP for Logistic Regression...")

explainer_lr_lifestyle = shap.LinearExplainer(
    lr_lifestyle,
    X_train_lifestyle_processed
)

shap_values_lr_lifestyle = (
    explainer_lr_lifestyle(
        X_val_lifestyle_processed
    )
)

print("Logistic Regression SHAP completed.")


shap.summary_plot(
    shap_values_lr_lifestyle.values,
    X_val_lifestyle_processed,
    feature_names=pcos_lifestyle_features,
    show=False
)

plt.title(
    "PCOS Lifestyle - Logistic Regression SHAP"
)

plt.tight_layout()

plt.show()

shap.summary_plot(
    shap_values_lr_lifestyle.values,
    X_val_lifestyle_processed,
    feature_names=pcos_lifestyle_features,
    plot_type="bar",
    show=False
)

plt.title(
    "PCOS Lifestyle - Logistic Regression SHAP Importance"
)

plt.tight_layout()

plt.show()

print("\nRunning SHAP for LANN...")

try:

    background_size_lifestyle = min(
        100,
        len(X_train_lifestyle_processed)
    )

    background_lifestyle = (
        X_train_lifestyle_processed[
            :background_size_lifestyle
        ]
    )

    explainer_lann_lifestyle = shap.Explainer(
        lann_lifestyle,
        background_lifestyle
    )

    shap_values_lann_lifestyle = (
        explainer_lann_lifestyle(
            X_val_lifestyle_processed
        )
    )

    print("LANN SHAP completed.")

    shap.summary_plot(
        shap_values_lann_lifestyle.values,
        X_val_lifestyle_processed,
        feature_names=pcos_lifestyle_features,
        show=False
    )

    plt.title(
        "PCOS Lifestyle LANN - SHAP Feature Importance"
    )

    plt.tight_layout()

    plt.show()
    shap.summary_plot(
        shap_values_lann_lifestyle.values,
        X_val_lifestyle_processed,
        feature_names=pcos_lifestyle_features,
        plot_type="bar",
        show=False
    )

    plt.title(
        "PCOS Lifestyle LANN - Mean Absolute SHAP Values"
    )

    plt.tight_layout()

    plt.show()

except Exception as e:

    print("\nLANN SHAP could not be generated.")
    print("Exact error:")
    print(e)
print("\n" + "=" * 70)
print("8. FINAL MODEL COMPARISON")
print("=" * 70)

final_comparison = lifestyle_test_report.copy()

print(
    final_comparison.to_string(index=False)
)

print("\n" + "=" * 70)
print("9. HIGHEST TEST METRICS")
print("=" * 70)

for metric in [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC"
]:

    best_index = lifestyle_test_df[
        metric
    ].idxmax()

    best_model = lifestyle_test_df.loc[
        best_index,
        "Model"
    ]

    best_value = lifestyle_test_df.loc[
        best_index,
        metric
    ]

    print(
        f"{metric}: "
        f"{best_model} = "
        f"{best_value:.4f}"
    )

print("\n" + "=" * 70)
print("10. SAVING RESULTS")
print("=" * 70)

lifestyle_test_report.to_csv(
    "PCOS_Lifestyle_Test_Results.csv",
    index=False
)

attention_lifestyle_df.to_csv(
    "PCOS_Lifestyle_LANN_Attention.csv",
    index=False
)

roc_results_df.to_csv(
    "PCOS_Lifestyle_ROC_AUC.csv",
    index=False
)

print(
    "Saved:"
)

print(
    "PCOS_Lifestyle_Test_Results.csv"
)

print(
    "PCOS_Lifestyle_LANN_Attention.csv"
)

print(
    "PCOS_Lifestyle_ROC_AUC.csv"
)

print("\n" + "=" * 70)
print("PCOS LIFESTYLE ANALYSIS COMPLETED")
print("=" * 70)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve
)

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    Multiply
)
from tensorflow.keras.callbacks import EarlyStopping

print("=" * 70)
print("THYROID EXPERIMENT")
print("=" * 70)

thyroid_df = df[
    df["thyroid_target"].notna()
].copy()

print("\n1. THYROID DATASET")
print("-" * 70)

print("Shape:", thyroid_df.shape)

print(
    "\nTarget distribution:"
)

print(
    thyroid_df["thyroid_target"]
    .value_counts()
    .sort_index()
)

thyroid_features = [
    "age",
    "sex"
]

thyroid_model_df = thyroid_df[
    thyroid_features
    + [
        "thyroid_target",
        "source_dataset",
        "source_patient_id"
    ]
].copy()

print("\nSelected features:")
print(thyroid_features)

print("\n2. DATA CLEANING")
print("-" * 70)

print(
    "\nAge before cleaning:"
)

print(
    thyroid_model_df["age"]
    .describe()
)

thyroid_model_df.loc[
    (thyroid_model_df["age"] < 18) |
    (thyroid_model_df["age"] > 100),
    "age"
] = None

thyroid_model_df["sex"] = (
    thyroid_model_df["sex"]
    .replace({
        "F": "Female",
        "Female": "Female",
        "M": "Male",
        "Male": "Male"
    })
)

print(
    "\nSex values after normalization:"
)

print(
    thyroid_model_df["sex"]
    .value_counts(dropna=False)
)

thyroid_model_df["patient_key"] = (
    thyroid_model_df["source_dataset"]
    .astype(str)
    + "_"
    + thyroid_model_df["source_patient_id"]
    .astype(str)
)

print("\n3. PATIENT KEY / LEAKAGE CHECK")
print("-" * 70)

print(
    "Total rows:",
    len(thyroid_model_df)
)

print(
    "Unique patient keys:",
    thyroid_model_df["patient_key"].nunique()
)

print(
    "Duplicated patient keys:",
    thyroid_model_df["patient_key"].duplicated().sum()
)

print(
    "Maximum rows per patient key:",
    thyroid_model_df["patient_key"].value_counts().max()
)

print(
    "\nTarget distribution:"
)

print(
    thyroid_model_df["thyroid_target"]
    .value_counts()
    .sort_index()
)

X = thyroid_model_df[
    thyroid_features
].copy()

y = thyroid_model_df[
    "thyroid_target"
].astype(int)

patient_keys = thyroid_model_df[
    "patient_key"
].copy()

print("\n4. TRAIN / VALIDATION / TEST SPLIT")
print("-" * 70)

X_train, X_temp, y_train, y_temp, key_train, key_temp = (
    train_test_split(
        X,
        y,
        patient_keys,
        test_size=0.30,
        stratify=y,
        random_state=42
    )
)

X_val, X_test, y_val, y_test, key_val, key_test = (
    train_test_split(
        X_temp,
        y_temp,
        key_temp,
        test_size=0.50,
        stratify=y_temp,
        random_state=42
    )
)

print(
    "Training:",
    X_train.shape
)

print(
    "Validation:",
    X_val.shape
)

print(
    "Test:",
    X_test.shape
)

print(
    "\nTarget distribution:"
)

print(
    "Train:"
)

print(
    y_train.value_counts()
    .sort_index()
)

print(
    "Validation:"
)

print(
    y_val.value_counts()
    .sort_index()
)

print(
    "Test:"
)

print(
    y_test.value_counts()
    .sort_index()
)

print(
    "\nLeakage checks:"
)

print(
    "Train ∩ Validation:",
    len(set(key_train) & set(key_val))
)

print(
    "Train ∩ Test:",
    len(set(key_train) & set(key_test))
)

print(
    "Validation ∩ Test:",
    len(set(key_val) & set(key_test))
)

print("\n5. PREPROCESSING")
print("-" * 70)

numeric_features = [
    "age"
]

categorical_features = [
    "sex"
]

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),
    (
        "scaler",
        StandardScaler()
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

thyroid_preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_features
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
])

X_train_processed = (
    thyroid_preprocessor.fit_transform(
        X_train
    )
)

X_val_processed = (
    thyroid_preprocessor.transform(
        X_val
    )
)

X_test_processed = (
    thyroid_preprocessor.transform(
        X_test
    )
)

if hasattr(
    X_train_processed,
    "toarray"
):
    X_train_processed = (
        X_train_processed.toarray()
    )

if hasattr(
    X_val_processed,
    "toarray"
):
    X_val_processed = (
        X_val_processed.toarray()
    )

if hasattr(
    X_test_processed,
    "toarray"
):
    X_test_processed = (
        X_test_processed.toarray()
    )

feature_names = (
    thyroid_preprocessor
    .get_feature_names_out()
)

print(
    "Processed train shape:",
    X_train_processed.shape
)

print(
    "Processed validation shape:",
    X_val_processed.shape
)

print(
    "Processed test shape:",
    X_test_processed.shape
)

print(
    "\nProcessed features:"
)

for i, feature in enumerate(
    feature_names
):
    print(
        i,
        feature
    )

print("\n6. TRAINING TRADITIONAL MODELS")
print("-" * 70)

lr_thyroid = LogisticRegression(
    max_iter=1000,
    random_state=42
)

dt_thyroid = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

rf_thyroid = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

xgb_thyroid = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

traditional_thyroid_models = {
    "Logistic Regression": lr_thyroid,
    "Decision Tree": dt_thyroid,
    "Random Forest": rf_thyroid,
    "XGBoost": xgb_thyroid
}

for model_name, model in (
    traditional_thyroid_models.items()
):

    print(
        "\nTraining:",
        model_name
    )

    model.fit(
        X_train_processed,
        y_train
    )

print("\n7. TRAINING STANDARD DNN")
print("-" * 70)

tf.random.set_seed(42)

dnn_thyroid = Sequential([
    Input(
        shape=(
            X_train_processed.shape[1],
        )
    ),
    Dense(
        64,
        activation="relu"
    ),
    Dropout(0.2),
    Dense(
        32,
        activation="relu"
    ),
    Dropout(0.2),
    Dense(
        1,
        activation="sigmoid"
    )
])

dnn_thyroid.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping_dnn = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history_dnn_thyroid = (
    dnn_thyroid.fit(
        X_train_processed,
        y_train,
        validation_data=(
            X_val_processed,
            y_val
        ),
        epochs=100,
        batch_size=32,
        callbacks=[
            early_stopping_dnn
        ],
        verbose=1
    )
)

print("\n8. TRAINING LANN")
print("-" * 70)

tf.random.set_seed(42)

input_dim_thyroid = (
    X_train_processed.shape[1]
)

inputs_thyroid = Input(
    shape=(input_dim_thyroid,)
)

attention_scores_thyroid = Dense(
    input_dim_thyroid,
    activation="softmax",
    name="feature_attention_thyroid"
)(
    inputs_thyroid
)

attended_features_thyroid = Multiply()([
    inputs_thyroid,
    attention_scores_thyroid
])

x_thyroid = Dense(
    64,
    activation="relu"
)(
    attended_features_thyroid
)

x_thyroid = Dropout(
    0.2
)(
    x_thyroid
)

x_thyroid = Dense(
    32,
    activation="relu"
)(
    x_thyroid
)

x_thyroid = Dropout(
    0.2
)(
    x_thyroid
)

outputs_thyroid = Dense(
    1,
    activation="sigmoid"
)(
    x_thyroid
)

lann_thyroid = Model(
    inputs=inputs_thyroid,
    outputs=outputs_thyroid,
    name="Thyroid_Lifestyle_Attention_Neural_Network"
)

lann_thyroid.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping_lann = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history_lann_thyroid = (
    lann_thyroid.fit(
        X_train_processed,
        y_train,
        validation_data=(
            X_val_processed,
            y_val
        ),
        epochs=100,
        batch_size=32,
        callbacks=[
            early_stopping_lann
        ],
        verbose=1
    )
)

def evaluate_thyroid_model(
    model,
    X_data,
    y_data,
    model_type="traditional"
):

    if model_type == "traditional":

        y_prob = (
            model.predict_proba(
                X_data
            )[:, 1]
        )

    else:

        y_prob = (
            model.predict(
                X_data,
                verbose=0
            ).ravel()
        )

    y_pred = (
        y_prob >= 0.5
    ).astype(int)

    return {
        "Accuracy": accuracy_score(
            y_data,
            y_pred
        ),
        "Precision": precision_score(
            y_data,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_data,
            y_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_data,
            y_pred,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_data,
            y_prob
        )
    }, y_pred, y_prob


print("\n9. VALIDATION RESULTS")
print("-" * 70)

thyroid_validation_results = []
thyroid_validation_predictions = {}

for model_name, model in (
    traditional_thyroid_models.items()
):

    metrics, y_pred, y_prob = (
        evaluate_thyroid_model(
            model,
            X_val_processed,
            y_val,
            "traditional"
        )
    )

    thyroid_validation_results.append({
        "Model": model_name,
        **metrics
    })

    thyroid_validation_predictions[
        model_name
    ] = (
        y_val.values,
        y_pred,
        y_prob
    )

metrics, y_pred, y_prob = (
    evaluate_thyroid_model(
        dnn_thyroid,
        X_val_processed,
        y_val,
        "neural"
    )
)

thyroid_validation_results.append({
    "Model": "Standard DNN",
    **metrics
})

thyroid_validation_predictions[
    "Standard DNN"
] = (
    y_val.values,
    y_pred,
    y_prob
)

metrics, y_pred, y_prob = (
    evaluate_thyroid_model(
        lann_thyroid,
        X_val_processed,
        y_val,
        "neural"
    )
)

thyroid_validation_results.append({
    "Model": "LANN",
    **metrics
})

thyroid_validation_predictions[
    "LANN"
] = (
    y_val.values,
    y_pred,
    y_prob
)

thyroid_validation_df = pd.DataFrame(
    thyroid_validation_results
)

print(
    thyroid_validation_df.round(4)
    .to_string(index=False)
)

print("\n10. FINAL TEST RESULTS")
print("-" * 70)

thyroid_test_results = []
thyroid_test_predictions = {}

for model_name, model in (
    traditional_thyroid_models.items()
):

    metrics, y_pred, y_prob = (
        evaluate_thyroid_model(
            model,
            X_test_processed,
            y_test,
            "traditional"
        )
    )

    thyroid_test_results.append({
        "Model": model_name,
        **metrics
    })

    thyroid_test_predictions[
        model_name
    ] = (
        y_test.values,
        y_pred,
        y_prob
    )

metrics, y_pred, y_prob = (
    evaluate_thyroid_model(
        dnn_thyroid,
        X_test_processed,
        y_test,
        "neural"
    )
)

thyroid_test_results.append({
    "Model": "Standard DNN",
    **metrics
})

thyroid_test_predictions[
    "Standard DNN"
] = (
    y_test.values,
    y_pred,
    y_prob
)

metrics, y_pred, y_prob = (
    evaluate_thyroid_model(
        lann_thyroid,
        X_test_processed,
        y_test,
        "neural"
    )
)

thyroid_test_results.append({
    "Model": "LANN",
    **metrics
})

thyroid_test_predictions[
    "LANN"
] = (
    y_test.values,
    y_pred,
    y_prob
)

thyroid_test_df = pd.DataFrame(
    thyroid_test_results
)

print(
    thyroid_test_df.round(4)
    .to_string(index=False)
)

print("\n11. TEST RESULTS - PERCENTAGE")
print("-" * 70)

thyroid_test_report = (
    thyroid_test_df.copy()
)

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC"
]

thyroid_test_report[
    metric_columns
] = (
    thyroid_test_report[
        metric_columns
    ] * 100
).round(2)

print(
    thyroid_test_report
    .to_string(index=False)
)

print("\n12. CONFUSION MATRICES")
print("-" * 70)

thyroid_confusion_results = {}

for model_name, (
    y_true,
    y_pred,
    y_prob
) in thyroid_test_predictions.items():

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    thyroid_confusion_results[
        model_name
    ] = cm

    tn, fp, fn, tp = cm.ravel()

    print(
        "\n" + model_name
    )

    print(
        cm
    )

    print(
        f"TN = {tn}"
    )

    print(
        f"FP = {fp}"
    )

    print(
        f"FN = {fn}"
    )

    print(
        f"TP = {tp}"
    )

print("\n13. CONFUSION MATRIX PLOTS")
print("-" * 70)

for model_name, (
    y_true,
    y_pred,
    y_prob
) in thyroid_test_predictions.items():

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    fig, ax = plt.subplots(
        figsize=(5, 4)
    )

    display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[
            "No Thyroid",
            "Thyroid"
        ]
    )

    display.plot(
        ax=ax,
        values_format="d"
    )

    ax.set_title(
        f"Thyroid - {model_name}\nTest Confusion Matrix"
    )

    plt.tight_layout()

    plt.show()


print("\n14. ROC CURVE")
print("-" * 70)

plt.figure(
    figsize=(9, 7)
)

thyroid_roc_results = []

for model_name, (
    y_true,
    y_pred,
    y_prob
) in thyroid_test_predictions.items():

    fpr, tpr, thresholds = (
        roc_curve(
            y_true,
            y_prob
        )
    )

    auc_value = (
        roc_auc_score(
            y_true,
            y_prob
        )
    )

    thyroid_roc_results.append({
        "Model": model_name,
        "ROC-AUC": auc_value
    })

    plt.plot(
        fpr,
        tpr,
        label=(
            f"{model_name} "
            f"(AUC = {auc_value:.3f})"
        )
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "Thyroid Models - ROC Curves"
)

plt.legend()

plt.grid()

plt.tight_layout()

plt.show()

thyroid_roc_df = pd.DataFrame(
    thyroid_roc_results
)

print(
    "\nROC-AUC values:"
)

print(
    thyroid_roc_df
    .sort_values(
        "ROC-AUC",
        ascending=False
    )
    .to_string(index=False)
)

print("\n15. LANN FEATURE ATTENTION")
print("-" * 70)

attention_layer_thyroid = (
    lann_thyroid.get_layer(
        "feature_attention_thyroid"
    )
)

attention_weights_thyroid = (
    attention_layer_thyroid.get_weights()
)

attention_kernel_thyroid = (
    attention_weights_thyroid[0]
)

attention_bias_thyroid = (
    attention_weights_thyroid[1]
)

attention_logits_thyroid = (
    np.dot(
        X_val_processed,
        attention_kernel_thyroid
    )
    + attention_bias_thyroid
)

attention_scores_thyroid = np.exp(
    attention_logits_thyroid
)

attention_scores_thyroid = (
    attention_scores_thyroid
    /
    attention_scores_thyroid.sum(
        axis=1,
        keepdims=True
    )
)

mean_attention_thyroid = (
    attention_scores_thyroid.mean(
        axis=0
    )
)

attention_thyroid_df = pd.DataFrame({
    "Feature": feature_names,
    "Mean Attention": mean_attention_thyroid
})

attention_thyroid_df = (
    attention_thyroid_df
    .sort_values(
        "Mean Attention",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    attention_thyroid_df.round(6)
    .to_string(index=False)
)

plt.figure(
    figsize=(9, 5)
)

plt.barh(
    attention_thyroid_df["Feature"],
    attention_thyroid_df["Mean Attention"]
)

plt.xlabel(
    "Mean Attention Weight"
)

plt.ylabel(
    "Feature"
)

plt.title(
    "Thyroid LANN - Feature Attention"
)

plt.gca().invert_yaxis()

plt.tight_layout()

plt.show()

print("\n16. SHAP ANALYSIS")
print("-" * 70)

try:
    import shap
    print(
        "SHAP is already installed."
    )

except ImportError:
    print(
        "Installing SHAP..."
    )

    !pip install -q shap

    import shap

print(
    "\nRunning SHAP for Logistic Regression..."
)

explainer_lr_thyroid = (
    shap.LinearExplainer(
        lr_thyroid,
        X_train_processed
    )
)

shap_values_lr_thyroid = (
    explainer_lr_thyroid(
        X_val_processed
    )
)

print(
    "Logistic Regression SHAP completed."
)

shap.summary_plot(
    shap_values_lr_thyroid.values,
    X_val_processed,
    feature_names=feature_names,
    show=False
)

plt.title(
    "Thyroid - Logistic Regression SHAP"
)

plt.tight_layout()

plt.show()

shap.summary_plot(
    shap_values_lr_thyroid.values,
    X_val_processed,
    feature_names=feature_names,
    plot_type="bar",
    show=False
)

plt.title(
    "Thyroid - Logistic Regression SHAP Importance"
)

plt.tight_layout()

plt.show()

print(
    "\nRunning SHAP for LANN..."
)

try:

    background_size_thyroid = min(
        100,
        len(X_train_processed)
    )

    background_thyroid = (
        X_train_processed[
            :background_size_thyroid
        ]
    )

    explainer_lann_thyroid = (
        shap.Explainer(
            lann_thyroid,
            background_thyroid
        )
    )

    shap_values_lann_thyroid = (
        explainer_lann_thyroid(
            X_val_processed
        )
    )

    print(
        "LANN SHAP completed."
    )

    shap.summary_plot(
        shap_values_lann_thyroid.values,
        X_val_processed,
        feature_names=feature_names,
        show=False
    )

    plt.title(
        "Thyroid LANN - SHAP Feature Importance"
    )

    plt.tight_layout()

    plt.show()

    shap.summary_plot(
        shap_values_lann_thyroid.values,
        X_val_processed,
        feature_names=feature_names,
        plot_type="bar",
        show=False
    )

    plt.title(
        "Thyroid LANN - Mean Absolute SHAP Values"
    )

    plt.tight_layout()

    plt.show()

except Exception as e:

    print(
        "\nLANN SHAP could not be generated."
    )

    print(
        "Exact error:"
    )

    print(e)

print("\n17. HIGHEST TEST METRICS")
print("-" * 70)

for metric in [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC"
]:

    best_index = (
        thyroid_test_df[
            metric
        ].idxmax()
    )

    best_model = (
        thyroid_test_df.loc[
            best_index,
            "Model"
        ]
    )

    best_value = (
        thyroid_test_df.loc[
            best_index,
            metric
        ]
    )

    print(
        f"{metric}: "
        f"{best_model} = "
        f"{best_value:.4f}"
    )


print("\n18. SAVING RESULTS")
print("-" * 70)

thyroid_validation_df.to_csv(
    "Thyroid_Validation_Results.csv",
    index=False
)

thyroid_test_report.to_csv(
    "Thyroid_Test_Results.csv",
    index=False
)

thyroid_roc_df.to_csv(
    "Thyroid_ROC_AUC.csv",
    index=False
)

attention_thyroid_df.to_csv(
    "Thyroid_LANN_Attention.csv",
    index=False
)

print(
    "Saved:"
)

print(
    "Thyroid_Validation_Results.csv"
)

print(
    "Thyroid_Test_Results.csv"
)

print(
    "Thyroid_ROC_AUC.csv"
)

print(
    "Thyroid_LANN_Attention.csv"
)

print("\n" + "=" * 70)
print("THYROID EXPERIMENT COMPLETED")
print("=" * 70)

In [ ]:
import pandas as pd
import os

print("=" * 70)
print("RECREATING DIABETES RESULT FILES")
print("=" * 70)

diabetes_test_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost",
        "Standard DNN",
        "LANN"
    ],
    "Accuracy": [
        0.7565,
        0.7130,
        0.7478,
        0.7304,
        0.7652,
        0.7391
    ],
    "Precision": [
        0.7544,
        0.8158,
        0.7258,
        0.7031,
        0.7586,
        0.7288
    ],
    "Recall": [
        0.7544,
        0.5439,
        0.7895,
        0.7895,
        0.7719,
        0.7544
    ],
    "F1": [
        0.7544,
        0.6526,
        0.7563,
        0.7438,
        0.7652,
        0.7414
    ],
    "ROC-AUC": [
        0.8439,
        0.7548,
        0.8566,
        0.8385,
        0.8627,
        0.8221
    ]
})

diabetes_test_results.to_csv(
    "Diabetes_Test_Results.csv",
    index=False
)


diabetes_validation_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost",
        "Standard DNN",
        "LANN"
    ],
    "Accuracy": [
        0.7391,
        0.6522,
        0.6957,
        0.6348,
        0.7043,
        0.6957
    ],
    "Precision": [
        0.7500,
        0.6800,
        0.7091,
        0.6481,
        0.7143,
        0.7170
    ],
    "Recall": [
        0.7241,
        0.5862,
        0.6724,
        0.6034,
        0.6897,
        0.6552
    ],
    "F1": [
        0.7368,
        0.6296,
        0.6903,
        0.6250,
        0.7018,
        0.6847
    ],
    "ROC-AUC": [
        0.7765,
        0.7281,
        0.7689,
        0.7287,
        0.7864,
        0.7635
    ]
})

diabetes_validation_results.to_csv(
    "Diabetes_Validation_Results.csv",
    index=False
)


diabetes_roc_auc = diabetes_test_results[
    ["Model", "ROC-AUC"]
].copy()

diabetes_roc_auc.to_csv(
    "Diabetes_ROC_AUC.csv",
    index=False
)


diabetes_attention = pd.DataFrame({
    "Feature": [
        "diet_type_Moderate",
        "diet_type_Unhealthy",
        "bmi",
        "smoking_0.0",
        "stress_score",
        "exercise_frequency_Rarely",
        "age",
        "exercise_frequency_1-2",
        "sleep_hours",
        "sex_Male",
        "exercise_frequency_Daily",
        "diet_type_Healthy",
        "exercise_frequency_3-5",
        "smoking_1.0",
        "sex_Female"
    ],
    "Mean Attention": [
        0.149377,
        0.145029,
        0.107341,
        0.068280,
        0.066283,
        0.063953,
        0.063523,
        0.061556,
        0.052665,
        0.039580,
        0.038922,
        0.036454,
        0.036032,
        0.035906,
        0.035101
    ]
})

diabetes_attention.to_csv(
    "Diabetes_LANN_Attention.csv",
    index=False
)


print("\nCreated files:")

for file in [
    "Diabetes_Test_Results.csv",
    "Diabetes_Validation_Results.csv",
    "Diabetes_ROC_AUC.csv",
    "Diabetes_LANN_Attention.csv"
]:
    print("FOUND:", file)

print("\nDiabetes result files recreated successfully.")

In [ ]:

import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

output_dir = "DA2_Final_Results"
os.makedirs(output_dir, exist_ok=True)

print("=" * 70)
print("DA2 FINAL COMBINED ANALYSIS")
print("=" * 70)


files_needed = [
    "Diabetes_Test_Results.csv",
    "Diabetes_Validation_Results.csv",
    "Diabetes_ROC_AUC.csv",
    "Diabetes_LANN_Attention.csv",
    "PCOS_Lifestyle_Test_Results.csv",
    "PCOS_Lifestyle_ROC_AUC.csv",
    "PCOS_Lifestyle_LANN_Attention.csv",
    "Thyroid_Test_Results.csv",
    "Thyroid_Validation_Results.csv",
    "Thyroid_ROC_AUC.csv",
    "Thyroid_LANN_Attention.csv"
]

print("\nChecking result files...")
print("-" * 70)

for file in files_needed:
    if os.path.exists(file):
        print("FOUND:", file)
    else:
        print("MISSING:", file)

def load_csv(filename):
    path = filename

    if not os.path.exists(path):
        matches = glob.glob("**/" + filename, recursive=True)

        if len(matches) > 0:
            path = matches[0]
        else:
            raise FileNotFoundError(
                f"Could not find {filename}. "
                f"Please make sure the corresponding experiment was run."
            )

    return pd.read_csv(path)

diabetes_test = load_csv("Diabetes_Test_Results.csv")
pcos_test = load_csv("PCOS_Lifestyle_Test_Results.csv")
thyroid_test = load_csv("Thyroid_Test_Results.csv")

print("\nLoaded test result tables.")
print("Diabetes:", diabetes_test.shape)
print("PCOS:", pcos_test.shape)
print("Thyroid:", thyroid_test.shape)

def standardize_result_table(df):
    df = df.copy()

    df.columns = [
        str(c).strip().replace("%", "").replace("-", "_")
        for c in df.columns
    ]

    rename_map = {}

    for c in df.columns:
        lower = c.lower()

        if lower == "model":
            rename_map[c] = "Model"

        elif lower == "accuracy":
            rename_map[c] = "Accuracy"

        elif lower == "precision":
            rename_map[c] = "Precision"

        elif lower == "recall":
            rename_map[c] = "Recall"

        elif lower == "f1":
            rename_map[c] = "F1"

        elif lower in ["roc_auc", "rocauc", "roc_auc_score"]:
            rename_map[c] = "ROC-AUC"

    df = df.rename(columns=rename_map)

    return df


diabetes_test = standardize_result_table(diabetes_test)
pcos_test = standardize_result_table(pcos_test)
thyroid_test = standardize_result_table(thyroid_test)

diabetes_test["Disease"] = "Diabetes"
pcos_test["Disease"] = "PCOS"
thyroid_test["Disease"] = "Thyroid"

diabetes_test["Experiment"] = "Diabetes Lifestyle"
pcos_test["Experiment"] = "PCOS Lifestyle"
thyroid_test["Experiment"] = "Thyroid Demographic"

combined_results = pd.concat(
    [
        diabetes_test,
        pcos_test,
        thyroid_test
    ],
    ignore_index=True
)

column_order = [
    "Disease",
    "Experiment",
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC"
]

combined_results = combined_results[
    [c for c in column_order if c in combined_results.columns]
]

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC"
]

for col in metric_columns:
    if col in combined_results.columns:
        combined_results[col] = pd.to_numeric(
            combined_results[col],
            errors="coerce"
        )


print("\n" + "=" * 70)
print("FINAL COMBINED TEST RESULTS")
print("=" * 70)

print(
    combined_results.to_string(
        index=False,
        formatters={
            "Accuracy": lambda x: f"{x:.4f}",
            "Precision": lambda x: f"{x:.4f}",
            "Recall": lambda x: f"{x:.4f}",
            "F1": lambda x: f"{x:.4f}",
            "ROC-AUC": lambda x: f"{x:.4f}"
        }
    )
)


combined_results.to_csv(
    os.path.join(output_dir, "All_Disease_Test_Results.csv"),
    index=False
)


percentage_results = combined_results.copy()

for col in metric_columns:
    percentage_results[col] = percentage_results[col] * 100

percentage_results.to_csv(
    os.path.join(output_dir, "All_Disease_Test_Results_Percentage.csv"),
    index=False
)

print("\nSaved:")
print("All_Disease_Test_Results.csv")
print("All_Disease_Test_Results_Percentage.csv")

print("\n" + "=" * 70)
print("DISEASE-WISE ACCURACY")
print("=" * 70)

accuracy_table = combined_results.pivot(
    index="Model",
    columns="Disease",
    values="Accuracy"
)

print(
    (accuracy_table * 100).round(2).to_string()
)

accuracy_table.to_csv(
    os.path.join(output_dir, "Accuracy_Comparison_Table.csv")
)

print("\n" + "=" * 70)
print("DISEASE-WISE F1 SCORE")
print("=" * 70)

f1_table = combined_results.pivot(
    index="Model",
    columns="Disease",
    values="F1"
)

print(
    (f1_table * 100).round(2).to_string()
)

f1_table.to_csv(
    os.path.join(output_dir, "F1_Comparison_Table.csv")
)

print("\n" + "=" * 70)
print("DISEASE-WISE ROC-AUC")
print("=" * 70)

auc_table = combined_results.pivot(
    index="Model",
    columns="Disease",
    values="ROC-AUC"
)

print(
    (auc_table * 100).round(2).to_string()
)

auc_table.to_csv(
    os.path.join(output_dir, "ROC_AUC_Comparison_Table.csv")
)

print("\n" + "=" * 70)
print("HIGHEST OBSERVED TEST METRICS")
print("=" * 70)

best_rows = []

for disease in combined_results["Disease"].unique():

    disease_df = combined_results[
        combined_results["Disease"] == disease
    ].copy()

    for metric in metric_columns:

        idx = disease_df[metric].idxmax()

        row = disease_df.loc[idx]

        best_rows.append({
            "Disease": disease,
            "Metric": metric,
            "Model": row["Model"],
            "Value": row[metric]
        })

best_metrics = pd.DataFrame(best_rows)

print(
    best_metrics.to_string(
        index=False,
        formatters={
            "Value": lambda x: f"{x:.4f}"
        }
    )
)

best_metrics.to_csv(
    os.path.join(output_dir, "Highest_Test_Metrics.csv"),
    index=False
)

print("\n" + "=" * 70)
print("LANN VS BEST BASELINE")
print("=" * 70)

lann_comparison = []

for disease in combined_results["Disease"].unique():

    disease_df = combined_results[
        combined_results["Disease"] == disease
    ].copy()

    lann_row = disease_df[
        disease_df["Model"].str.contains(
            "LANN",
            case=False,
            na=False
        )
    ]

    baseline_df = disease_df[
        ~disease_df["Model"].str.contains(
            "LANN",
            case=False,
            na=False
        )
    ]

    if len(lann_row) == 0:
        continue

    lann_row = lann_row.iloc[0]

    for metric in metric_columns:

        best_idx = baseline_df[metric].idxmax()
        best_baseline = baseline_df.loc[best_idx]

        lann_comparison.append({
            "Disease": disease,
            "Metric": metric,
            "LANN": lann_row[metric],
            "Best_Baseline": best_baseline[metric],
            "Best_Baseline_Model": best_baseline["Model"],
            "Difference_LANN_minus_Baseline":
                lann_row[metric] - best_baseline[metric]
        })

lann_comparison_df = pd.DataFrame(lann_comparison)

print(
    lann_comparison_df.to_string(
        index=False,
        formatters={
            "LANN": lambda x: f"{x:.4f}",
            "Best_Baseline": lambda x: f"{x:.4f}",
            "Difference_LANN_minus_Baseline":
                lambda x: f"{x:+.4f}"
        }
    )
)

lann_comparison_df.to_csv(
    os.path.join(output_dir, "LANN_vs_Best_Baseline.csv"),
    index=False
)

plt.figure(figsize=(12, 7))

sns.barplot(
    data=combined_results,
    x="Disease",
    y="Accuracy",
    hue="Model"
)

plt.title("Accuracy Comparison Across Diseases")
plt.ylabel("Accuracy")
plt.xlabel("Disease")
plt.ylim(0, 1)
plt.legend(
    title="Model",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout()

plt.savefig(
    os.path.join(output_dir, "Accuracy_Comparison.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.figure(figsize=(12, 7))

sns.barplot(
    data=combined_results,
    x="Disease",
    y="Precision",
    hue="Model"
)

plt.title("Precision Comparison Across Diseases")
plt.ylabel("Precision")
plt.xlabel("Disease")
plt.ylim(0, 1)
plt.legend(
    title="Model",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout()

plt.savefig(
    os.path.join(output_dir, "Precision_Comparison.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.figure(figsize=(12, 7))

sns.barplot(
    data=combined_results,
    x="Disease",
    y="Recall",
    hue="Model"
)

plt.title("Recall Comparison Across Diseases")
plt.ylabel("Recall")
plt.xlabel("Disease")
plt.ylim(0, 1)
plt.legend(
    title="Model",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout()

plt.savefig(
    os.path.join(output_dir, "Recall_Comparison.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.figure(figsize=(12, 7))

sns.barplot(
    data=combined_results,
    x="Disease",
    y="F1",
    hue="Model"
)

plt.title("F1-Score Comparison Across Diseases")
plt.ylabel("F1-Score")
plt.xlabel("Disease")
plt.ylim(0, 1)
plt.legend(
    title="Model",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout()

plt.savefig(
    os.path.join(output_dir, "F1_Comparison.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.figure(figsize=(12, 7))

sns.barplot(
    data=combined_results,
    x="Disease",
    y="ROC-AUC",
    hue="Model"
)

plt.title("ROC-AUC Comparison Across Diseases")
plt.ylabel("ROC-AUC")
plt.xlabel("Disease")
plt.ylim(0, 1)
plt.legend(
    title="Model",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout()

plt.savefig(
    os.path.join(output_dir, "ROC_AUC_Comparison.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

lann_results = combined_results[
    combined_results["Model"].str.contains(
        "LANN",
        case=False,
        na=False
    )
].copy()

print("\n" + "=" * 70)
print("LANN RESULTS ACROSS ALL EXPERIMENTS")
print("=" * 70)

print(
    lann_results.to_string(
        index=False,
        formatters={
            "Accuracy": lambda x: f"{x:.4f}",
            "Precision": lambda x: f"{x:.4f}",
            "Recall": lambda x: f"{x:.4f}",
            "F1": lambda x: f"{x:.4f}",
            "ROC-AUC": lambda x: f"{x:.4f}"
        }
    )
)

lann_results.to_csv(
    os.path.join(output_dir, "LANN_All_Diseases.csv"),
    index=False
)

lann_plot_df = lann_results.melt(
    id_vars=["Disease"],
    value_vars=metric_columns,
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(11, 7))

sns.barplot(
    data=lann_plot_df,
    x="Disease",
    y="Score",
    hue="Metric"
)

plt.title("LANN Performance Across Diseases")
plt.ylabel("Score")
plt.xlabel("Disease")
plt.ylim(0, 1)
plt.legend(
    title="Metric",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout()

plt.savefig(
    os.path.join(output_dir, "LANN_Performance.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

attention_tables = []

attention_files = [
    ("Diabetes", "Diabetes_LANN_Attention.csv"),
    ("PCOS", "PCOS_Lifestyle_LANN_Attention.csv"),
    ("Thyroid", "Thyroid_LANN_Attention.csv")
]

for disease, filename in attention_files:

    try:
        temp = load_csv(filename)

        temp = temp.copy()

        temp["Disease"] = disease

        attention_tables.append(temp)

    except Exception as e:
        print(
            f"Could not load attention file {filename}: {e}"
        )

if len(attention_tables) > 0:

    attention_combined = pd.concat(
        attention_tables,
        ignore_index=True
    )

    print("\n" + "=" * 70)
    print("COMBINED LANN ATTENTION")
    print("=" * 70)

    print(
        attention_combined.to_string(
            index=False
        )
    )

    attention_combined.to_csv(
        os.path.join(
            output_dir,
            "All_Disease_LANN_Attention.csv"
        ),
        index=False
    )

else:

    attention_combined = None

    print("\nNo attention files were found.")

if attention_combined is not None:

    for disease in attention_combined["Disease"].unique():

        disease_attention = attention_combined[
            attention_combined["Disease"] == disease
        ].copy()

        if "Mean Attention" in disease_attention.columns:

            disease_attention = disease_attention.sort_values(
                "Mean Attention",
                ascending=False
            )

            plt.figure(figsize=(10, 7))

            sns.barplot(
                data=disease_attention,
                x="Mean Attention",
                y="Feature"
            )

            plt.title(
                f"LANN Feature Attention - {disease}"
            )

            plt.xlabel("Mean Attention")
            plt.ylabel("Feature")

            plt.tight_layout()

            safe_name = (
                disease
                .replace(" ", "_")
                .replace("/", "_")
            )

            plt.savefig(
                os.path.join(
                    output_dir,
                    f"{safe_name}_LANN_Attention.png"
                ),
                dpi=300,
                bbox_inches="tight"
            )

            plt.show()


dataset_summary = pd.DataFrame({
    "Disease": [
        "Diabetes",
        "PCOS",
        "Thyroid"
    ],
    "Samples": [
        766,
        766,
        766
    ],
    "Positive": [
        383,
        383,
        383
    ],
    "Negative": [
        383,
        383,
        383
    ],
    "Split_Train": [
        536,
        536,
        536
    ],
    "Split_Validation": [
        115,
        115,
        115
    ],
    "Split_Test": [
        115,
        115,
        115
    ],
    "Experiment_Features": [
        "Age, BMI, Sex, Smoking, Sleep, Diet, Stress, Exercise",
        "Exercise, Fast Food, Physical Activity, Smoking, Sugar, Sleep",
        "Age, Sex"
    ]
})

print("\n" + "=" * 70)
print("DATASET SUMMARY")
print("=" * 70)

print(
    dataset_summary.to_string(index=False)
)

dataset_summary.to_csv(
    os.path.join(
        output_dir,
        "Dataset_Summary.csv"
    ),
    index=False
)


print("\n" + "=" * 70)
print("FINAL RESEARCH OBSERVATIONS")
print("=" * 70)

for disease in ["Diabetes", "PCOS", "Thyroid"]:

    disease_df = combined_results[
        combined_results["Disease"] == disease
    ].copy()

    print(f"\n{disease}")
    print("-" * 40)

    for metric in metric_columns:

        best_idx = disease_df[metric].idxmax()

        best_row = disease_df.loc[best_idx]

        print(
            f"{metric}: "
            f"{best_row['Model']} "
            f"({best_row[metric] * 100:.2f}%)"
        )


print("\n" + "=" * 70)
print("LANN COMPARISON SUMMARY")
print("=" * 70)

for disease in combined_results["Disease"].unique():

    disease_df = combined_results[
        combined_results["Disease"] == disease
    ]

    lann = disease_df[
        disease_df["Model"].str.contains(
            "LANN",
            case=False,
            na=False
        )
    ]

    baseline = disease_df[
        ~disease_df["Model"].str.contains(
            "LANN",
            case=False,
            na=False
        )
    ]

    if len(lann) == 0:
        continue

    lann = lann.iloc[0]

    print(f"\n{disease}")

    for metric in metric_columns:

        best_baseline_value = baseline[metric].max()

        difference = (
            lann[metric] -
            best_baseline_value
        )

        print(
            f"{metric}: "
            f"LANN={lann[metric] * 100:.2f}% | "
            f"Best baseline={best_baseline_value * 100:.2f}% | "
            f"Difference={difference * 100:+.2f}%"
        )

summary_path = os.path.join(
    output_dir,
    "Final_Research_Summary.txt"
)

with open(summary_path, "w", encoding="utf-8") as f:

    f.write("DA2 FINAL RESEARCH SUMMARY\n")
    f.write("=" * 70 + "\n\n")

    f.write(
        "Project: AI Predicting Hormonal Disorders Using "
        "Non-Invasive Lifestyle\n\n"
    )

    f.write("Dataset Summary\n")
    f.write("-" * 70 + "\n")

    for _, row in dataset_summary.iterrows():

        f.write(
            f"{row['Disease']}: "
            f"{row['Samples']} samples, "
            f"{row['Positive']} positive, "
            f"{row['Negative']} negative\n"
        )

    f.write("\nModel Results\n")
    f.write("-" * 70 + "\n")

    for disease in ["Diabetes", "PCOS", "Thyroid"]:

        f.write(f"\n{disease}\n")

        disease_df = combined_results[
            combined_results["Disease"] == disease
        ]

        for _, row in disease_df.iterrows():

            f.write(
                f"{row['Model']}: "
                f"Accuracy={row['Accuracy'] * 100:.2f}%, "
                f"Precision={row['Precision'] * 100:.2f}%, "
                f"Recall={row['Recall'] * 100:.2f}%, "
                f"F1={row['F1'] * 100:.2f}%, "
                f"ROC-AUC={row['ROC-AUC'] * 100:.2f}%\n"
            )

    f.write("\nImportant Interpretation\n")
    f.write("-" * 70 + "\n")

    f.write(
        "The models are evaluated separately for diabetes, PCOS, "
        "and thyroid classification because feature availability "
        "differs substantially across the source datasets.\n"
    )

    f.write(
        "No artificial lifestyle values were created for the thyroid "
        "dataset. The thyroid experiment therefore used age and sex, "
        "which were the available selected non-invasive features.\n"
    )

    f.write(
        "LANN attention values represent model emphasis and should "
        "not be interpreted as causal relationships.\n"
    )

    f.write(
        "The reported results are classification/screening results "
        "and should not be described as clinical diagnosis.\n"
    )

print("\n" + "=" * 70)
print("FINAL OUTPUT FILES")
print("=" * 70)

for filename in sorted(os.listdir(output_dir)):
    print(filename)

print("\n" + "=" * 70)
print("FINAL COMBINED ANALYSIS COMPLETED")
print("=" * 70)

print(
    "\nAll final tables, graphs, attention results, "
    "and summary files are inside:"
)

print(output_dir)

In [ ]:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

output_dir = "DA2_Final_Results"
os.makedirs(output_dir, exist_ok=True)

print("=" * 70)
print("FINAL COMBINED ANALYSIS - CORRECTED")
print("=" * 70)

diabetes = pd.read_csv("Diabetes_Test_Results.csv")
pcos = pd.read_csv("PCOS_Lifestyle_Test_Results.csv")
thyroid = pd.read_csv("Thyroid_Test_Results.csv")

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC"
]

print("\nOriginal PCOS values:")
print(pcos.to_string(index=False))

for col in metric_columns:
    if col in pcos.columns:
        pcos[col] = pd.to_numeric(
            pcos[col],
            errors="coerce"
        )

        pcos[col] = pcos[col] / 100.0

diabetes["Disease"] = "Diabetes"
diabetes["Experiment"] = "Diabetes Lifestyle"

pcos["Disease"] = "PCOS"
pcos["Experiment"] = "PCOS Lifestyle"

thyroid["Disease"] = "Thyroid"
thyroid["Experiment"] = "Thyroid Demographic"


combined = pd.concat(
    [
        diabetes,
        pcos,
        thyroid
    ],
    ignore_index=True
)

combined = combined[
    [
        "Disease",
        "Experiment",
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ]
]


for col in metric_columns:
    combined[col] = pd.to_numeric(
        combined[col],
        errors="coerce"
    )

print("\n" + "=" * 70)
print("CORRECTED FINAL TEST RESULTS")
print("=" * 70)

display_table = combined.copy()

for col in metric_columns:
    display_table[col] = (
        display_table[col] * 100
    ).round(2)

print(display_table.to_string(index=False))


combined.to_csv(
    os.path.join(
        output_dir,
        "All_Disease_Test_Results.csv"
    ),
    index=False
)


percentage = combined.copy()

for col in metric_columns:
    percentage[col] = (
        percentage[col] * 100
    ).round(2)

percentage.to_csv(
    os.path.join(
        output_dir,
        "All_Disease_Test_Results_Percentage.csv"
    ),
    index=False
)


accuracy_table = combined.pivot(
    index="Model",
    columns="Disease",
    values="Accuracy"
) * 100

accuracy_table = accuracy_table.round(2)

print("\n" + "=" * 70)
print("DISEASE-WISE ACCURACY (%)")
print("=" * 70)

print(accuracy_table.to_string())

accuracy_table.to_csv(
    os.path.join(
        output_dir,
        "Accuracy_Comparison_Table.csv"
    )
)


precision_table = combined.pivot(
    index="Model",
    columns="Disease",
    values="Precision"
) * 100

precision_table = precision_table.round(2)

print("\n" + "=" * 70)
print("DISEASE-WISE PRECISION (%)")
print("=" * 70)

print(precision_table.to_string())

precision_table.to_csv(
    os.path.join(
        output_dir,
        "Precision_Comparison_Table.csv"
    )
)


recall_table = combined.pivot(
    index="Model",
    columns="Disease",
    values="Recall"
) * 100

recall_table = recall_table.round(2)

print("\n" + "=" * 70)
print("DISEASE-WISE RECALL (%)")
print("=" * 70)

print(recall_table.to_string())

recall_table.to_csv(
    os.path.join(
        output_dir,
        "Recall_Comparison_Table.csv"
    )
)


f1_table = combined.pivot(
    index="Model",
    columns="Disease",
    values="F1"
) * 100

f1_table = f1_table.round(2)

print("\n" + "=" * 70)
print("DISEASE-WISE F1 (%)")
print("=" * 70)

print(f1_table.to_string())

f1_table.to_csv(
    os.path.join(
        output_dir,
        "F1_Comparison_Table.csv"
    )
)

auc_table = combined.pivot(
    index="Model",
    columns="Disease",
    values="ROC-AUC"
) * 100

auc_table = auc_table.round(2)

print("\n" + "=" * 70)
print("DISEASE-WISE ROC-AUC (%)")
print("=" * 70)

print(auc_table.to_string())

auc_table.to_csv(
    os.path.join(
        output_dir,
        "ROC_AUC_Comparison_Table.csv"
    )
)
best_rows = []

for disease in combined["Disease"].unique():

    disease_df = combined[
        combined["Disease"] == disease
    ]

    for metric in metric_columns:

        idx = disease_df[metric].idxmax()

        row = disease_df.loc[idx]

        best_rows.append({
            "Disease": disease,
            "Metric": metric,
            "Model": row["Model"],
            "Value": row[metric]
        })

best_metrics = pd.DataFrame(best_rows)

print("\n" + "=" * 70)
print("HIGHEST OBSERVED TEST METRICS")
print("=" * 70)

best_display = best_metrics.copy()
best_display["Value"] = (
    best_display["Value"] * 100
).round(2)

print(
    best_display.to_string(index=False)
)

best_metrics.to_csv(
    os.path.join(
        output_dir,
        "Highest_Test_Metrics.csv"
    ),
    index=False
)


comparison_rows = []

for disease in combined["Disease"].unique():

    disease_df = combined[
        combined["Disease"] == disease
    ]

    lann_df = disease_df[
        disease_df["Model"].str.lower() == "lann"
    ]

    baseline_df = disease_df[
        disease_df["Model"].str.lower() != "lann"
    ]

    if len(lann_df) == 0:
        continue

    lann = lann_df.iloc[0]

    for metric in metric_columns:

        idx = baseline_df[metric].idxmax()

        baseline = baseline_df.loc[idx]

        comparison_rows.append({
            "Disease": disease,
            "Metric": metric,
            "LANN": lann[metric],
            "Best_Baseline": baseline[metric],
            "Best_Baseline_Model": baseline["Model"],
            "Difference": (
                lann[metric] -
                baseline[metric]
            )
        })

lann_comparison = pd.DataFrame(
    comparison_rows
)

print("\n" + "=" * 70)
print("LANN VS BEST BASELINE")
print("=" * 70)

comparison_display = lann_comparison.copy()

for col in [
    "LANN",
    "Best_Baseline",
    "Difference"
]:
    comparison_display[col] = (
        comparison_display[col] * 100
    ).round(2)

print(
    comparison_display.to_string(index=False)
)

lann_comparison.to_csv(
    os.path.join(
        output_dir,
        "LANN_vs_Best_Baseline.csv"
    ),
    index=False
)

def create_metric_plot(
    data,
    metric,
    filename,
    title
):

    plt.figure(figsize=(12, 7))

    sns.barplot(
        data=data,
        x="Disease",
        y=metric,
        hue="Model"
    )

    plt.title(title)
    plt.xlabel("Disease")
    plt.ylabel(metric)
    plt.ylim(0, 1)

    plt.legend(
        title="Model",
        bbox_to_anchor=(1.02, 1),
        loc="upper left"
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            output_dir,
            filename
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

create_metric_plot(
    combined,
    "Accuracy",
    "Accuracy_Comparison.png",
    "Accuracy Comparison Across Diseases"
)

create_metric_plot(
    combined,
    "Precision",
    "Precision_Comparison.png",
    "Precision Comparison Across Diseases"
)

create_metric_plot(
    combined,
    "Recall",
    "Recall_Comparison.png",
    "Recall Comparison Across Diseases"
)

create_metric_plot(
    combined,
    "F1",
    "F1_Comparison.png",
    "F1-Score Comparison Across Diseases"
)

create_metric_plot(
    combined,
    "ROC-AUC",
    "ROC_AUC_Comparison.png",
    "ROC-AUC Comparison Across Diseases"
)


lann_results = combined[
    combined["Model"].str.lower() == "lann"
].copy()

print("\n" + "=" * 70)
print("LANN RESULTS ACROSS ALL DISEASES")
print("=" * 70)

lann_display = lann_results.copy()

for col in metric_columns:
    lann_display[col] = (
        lann_display[col] * 100
    ).round(2)

print(
    lann_display.to_string(index=False)
)

lann_results.to_csv(
    os.path.join(
        output_dir,
        "LANN_All_Diseases.csv"
    ),
    index=False
)

lann_plot = lann_results[
    ["Disease"] + metric_columns
].melt(
    id_vars=["Disease"],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(11, 7))

sns.barplot(
    data=lann_plot,
    x="Disease",
    y="Score",
    hue="Metric"
)

plt.title(
    "LANN Performance Across Diseases"
)

plt.xlabel("Disease")
plt.ylabel("Score")
plt.ylim(0, 1)

plt.legend(
    title="Metric",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        output_dir,
        "LANN_Performance.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()


attention_files = [
    ("Diabetes", "Diabetes_LANN_Attention.csv"),
    ("PCOS", "PCOS_Lifestyle_LANN_Attention.csv"),
    ("Thyroid", "Thyroid_LANN_Attention.csv")
]

attention_tables = []

for disease, filename in attention_files:

    if os.path.exists(filename):

        temp = pd.read_csv(filename)

        temp["Disease"] = disease

        attention_tables.append(temp)


if len(attention_tables) > 0:

    attention = pd.concat(
        attention_tables,
        ignore_index=True
    )

    print("\n" + "=" * 70)
    print("COMBINED LANN ATTENTION")
    print("=" * 70)

    print(
        attention.to_string(index=False)
    )

    attention.to_csv(
        os.path.join(
            output_dir,
            "All_Disease_LANN_Attention.csv"
        ),
        index=False
    )

    for disease in attention["Disease"].unique():

        temp = attention[
            attention["Disease"] == disease
        ].copy()

        if "Mean Attention" not in temp.columns:
            continue

        temp = temp.sort_values(
            "Mean Attention",
            ascending=False
        )

        plt.figure(figsize=(10, 7))

        sns.barplot(
            data=temp,
            x="Mean Attention",
            y="Feature"
        )

        plt.title(
            f"LANN Feature Attention - {disease}"
        )

        plt.xlabel("Mean Attention")
        plt.ylabel("Feature")

        plt.tight_layout()

        safe_name = disease.replace(
            " ",
            "_"
        )

        plt.savefig(
            os.path.join(
                output_dir,
                f"{safe_name}_LANN_Attention.png"
            ),
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()


dataset_summary = pd.DataFrame({
    "Disease": [
        "Diabetes",
        "PCOS",
        "Thyroid"
    ],
    "Samples": [
        766,
        766,
        766
    ],
    "Positive": [
        383,
        383,
        383
    ],
    "Negative": [
        383,
        383,
        383
    ],
    "Train": [
        536,
        536,
        536
    ],
    "Validation": [
        115,
        115,
        115
    ],
    "Test": [
        115,
        115,
        115
    ],
    "Features": [
        "Age, BMI, Sex, Smoking, Sleep, Diet, Stress, Exercise",
        "Exercise, Fast Food, Physical Activity, Smoking, Sugar, Sleep",
        "Age, Sex"
    ]
})

dataset_summary.to_csv(
    os.path.join(
        output_dir,
        "Dataset_Summary.csv"
    ),
    index=False
)

summary_file = os.path.join(
    output_dir,
    "Final_Research_Summary.txt"
)

with open(
    summary_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "DA2 FINAL RESEARCH SUMMARY\n"
    )

    f.write("=" * 70 + "\n\n")

    f.write(
        "Project: AI Predicting Hormonal Disorders "
        "Using Non-Invasive Lifestyle\n\n"
    )

    f.write(
        "IMPORTANT: Results represent classification/screening "
        "experiments and should not be interpreted as clinical "
        "diagnosis.\n\n"
    )

    f.write(
        "DATASET SUMMARY\n"
    )

    f.write("-" * 70 + "\n")

    for _, row in dataset_summary.iterrows():

        f.write(
            f"{row['Disease']}: "
            f"{row['Samples']} samples; "
            f"{row['Positive']} positive; "
            f"{row['Negative']} negative; "
            f"Train={row['Train']}; "
            f"Validation={row['Validation']}; "
            f"Test={row['Test']}\n"
        )

    f.write("\nMODEL RESULTS\n")
    f.write("-" * 70 + "\n")

    for disease in [
        "Diabetes",
        "PCOS",
        "Thyroid"
    ]:

        f.write(
            f"\n{disease}\n"
        )

        temp = combined[
            combined["Disease"] == disease
        ]

        for _, row in temp.iterrows():

            f.write(
                f"{row['Model']}: "
                f"Accuracy={row['Accuracy'] * 100:.2f}%, "
                f"Precision={row['Precision'] * 100:.2f}%, "
                f"Recall={row['Recall'] * 100:.2f}%, "
                f"F1={row['F1'] * 100:.2f}%, "
                f"ROC-AUC={row['ROC-AUC'] * 100:.2f}%\n"
            )

    f.write("\nKEY OBSERVATIONS\n")
    f.write("-" * 70 + "\n")

    f.write(
        "Diabetes: Standard DNN obtained the highest observed "
        "accuracy, F1-score and ROC-AUC among the evaluated models.\n"
    )

    f.write(
        "PCOS: XGBoost obtained the highest observed accuracy, "
        "precision and ROC-AUC, while LANN obtained the highest "
        "recall.\n"
    )

    f.write(
        "Thyroid: LANN obtained the highest observed accuracy, "
        "precision and F1-score, while overall ROC-AUC values "
        "remained close to chance-level discrimination.\n"
    )

    f.write(
        "LANN attention values indicate model emphasis on input "
        "features and should not be interpreted as causal effects.\n"
    )

    f.write(
        "The three disease experiments use different feature sets "
        "because feature availability differs across their source "
        "datasets.\n"
    )

    f.write(
        "No unavailable lifestyle variables were artificially "
        "generated for the thyroid experiment.\n"
    )


print("\n" + "=" * 70)
print("FINAL CORRECTED ANALYSIS COMPLETED")
print("=" * 70)

print("\nOutput folder:")
print(output_dir)

print("\nImportant corrected files:")

for filename in [
    "All_Disease_Test_Results.csv",
    "All_Disease_Test_Results_Percentage.csv",
    "Accuracy_Comparison_Table.csv",
    "Precision_Comparison_Table.csv",
    "Recall_Comparison_Table.csv",
    "F1_Comparison_Table.csv",
    "ROC_AUC_Comparison_Table.csv",
    "Highest_Test_Metrics.csv",
    "LANN_vs_Best_Baseline.csv",
    "LANN_All_Diseases.csv",
    "Final_Research_Summary.txt"
]:

    if os.path.exists(
        os.path.join(output_dir, filename)
    ):
        print("✓", filename)

print("\nALL RESULTS ARE NOW ON THE SAME 0-1 SCALE.")
print("Percentage tables contain 0-100 values.")
print("=" * 70)

In [ ]:
print("Current variables available:")

for name in [
    "df",
    "diabetes_df",
    "pcos_df",
    "pcos_lifestyle_df",
    "thyroid_df",
    "X_train",
    "X_val",
    "X_test",
    "y_train",
    "y_val",
    "y_test",
    "lann_model",
    "dnn_model",
    "preprocessor"
]:
    print(name, "->", "FOUND" if name in globals() else "NOT FOUND")

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Dense, Dropout, Multiply
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)

print("PCOS ABLATION STUDY")
print("=" * 60)

pcos_ablation_features = [
    "regular_exercise",
    "fast_food",
    "physical_activity_level",
    "smoking",
    "dietary_sugar_intake",
    "sleep_hours"
]

pcos_ablation_df = pcos_lifestyle_df[
    pcos_ablation_features + ["pcos_target"]
].copy()

print("Dataset shape:", pcos_ablation_df.shape)
print("\nMissing values:")
print(pcos_ablation_df.isnull().sum())

print("\nTarget distribution:")
print(pcos_ablation_df["pcos_target"].value_counts())

In [ ]:
X_ablation = pcos_ablation_df[pcos_ablation_features].copy()
y_ablation = pcos_ablation_df["pcos_target"].astype(int).copy()

X_ab_train, X_ab_temp, y_ab_train, y_ab_temp = train_test_split(
    X_ablation,
    y_ablation,
    test_size=0.30,
    stratify=y_ablation,
    random_state=42
)

X_ab_val, X_ab_test, y_ab_val, y_ab_test = train_test_split(
    X_ab_temp,
    y_ab_temp,
    test_size=0.50,
    stratify=y_ab_temp,
    random_state=42
)

print("Training:", X_ab_train.shape)
print("Validation:", X_ab_val.shape)
print("Testing:", X_ab_test.shape)

print("\nClass distributions:")
print("Train:")
print(y_ab_train.value_counts())

print("\nValidation:")
print(y_ab_val.value_counts())

print("\nTest:")
print(y_ab_test.value_counts())

In [ ]:
ablation_imputer = SimpleImputer(strategy="median")
ablation_scaler = StandardScaler()

X_ab_train_imp = ablation_imputer.fit_transform(X_ab_train)
X_ab_val_imp = ablation_imputer.transform(X_ab_val)
X_ab_test_imp = ablation_imputer.transform(X_ab_test)

X_ab_train_scaled = ablation_scaler.fit_transform(X_ab_train_imp)
X_ab_val_scaled = ablation_scaler.transform(X_ab_val_imp)
X_ab_test_scaled = ablation_scaler.transform(X_ab_test_imp)

print("Processed training shape:", X_ab_train_scaled.shape)
print("Processed validation shape:", X_ab_val_scaled.shape)
print("Processed test shape:", X_ab_test_scaled.shape)

In [ ]:
print("pcos_lifestyle_df shape:", pcos_lifestyle_df.shape)

print("\nColumns:")
print(pcos_lifestyle_df.columns.tolist())

print("\nMissing values:")
print(
    pcos_lifestyle_df[
        [
            "regular_exercise",
            "fast_food",
            "physical_activity_level",
            "smoking",
            "dietary_sugar_intake",
            "sleep_hours",
            "pcos_target"
        ]
    ].isnull().sum()
)

In [ ]:
pcos_lifestyle_features = [
    "regular_exercise",
    "fast_food",
    "physical_activity_level",
    "smoking",
    "dietary_sugar_intake",
    "sleep_hours"
]

pcos_ablation_df = pcos_lifestyle_df[
    pcos_lifestyle_features + ["pcos_target", "patient_key"]
].dropna(
    subset=pcos_lifestyle_features
).copy()

print("PCOS ablation dataset shape:", pcos_ablation_df.shape)

print("\nMissing values:")
print(pcos_ablation_df[pcos_lifestyle_features].isnull().sum())

print("\nTarget distribution:")
print(pcos_ablation_df["pcos_target"].value_counts())

print("\nUnique patients:")
print(pcos_ablation_df["patient_key"].nunique())

In [ ]:
print("Shape:", pcos_ablation_df.shape)

print("\nTarget values:")
print(pcos_ablation_df["pcos_target"].value_counts(dropna=False))

print("\nTarget dtype:")
print(pcos_ablation_df["pcos_target"].dtype)

print("\nLifestyle columns:")
for col in pcos_ablation_features:
    print("\n", col)
    print(pcos_ablation_df[col].value_counts(dropna=False).head(15))

In [ ]:
pcos_ablation_features = [
    "regular_exercise",
    "fast_food",
    "physical_activity_level",
    "smoking",
    "dietary_sugar_intake",
    "sleep_hours"
]

pcos_ablation_df = df[
    df["source_dataset"] == "kaggle_pcos_extended_patient_dedup"
].copy()

print("PCOS ablation dataset shape:", pcos_ablation_df.shape)

print("\nTarget distribution:")
print(pcos_ablation_df["pcos_target"].value_counts())

print("\nMissing values:")
print(pcos_ablation_df[pcos_ablation_features].isnull().sum())

print("\nSelected features:")
print(pcos_ablation_df[pcos_ablation_features].head())

In [ ]:
pcos_ablation_df = pcos_ablation_df[
    pcos_ablation_features + ["pcos_target", "source_dataset", "source_patient_id"]
].copy()

print("Final ablation dataset shape:", pcos_ablation_df.shape)

print("\nMissing values:")
print(pcos_ablation_df[pcos_ablation_features].isnull().sum())

print("\nTarget distribution:")
print(pcos_ablation_df["pcos_target"].value_counts())

In [ ]:
for col in ["regular_exercise", "fast_food"]:
    print("\n" + "=" * 50)
    print(col)
    print(pcos_ablation_df[col].value_counts(dropna=False))

In [ ]:
print(
    pcos_ablation_df[
        ["regular_exercise", "fast_food", "pcos_target"]
    ].head(20)
)

In [ ]:
pcos_ablation_features = [
    "regular_exercise",
    "fast_food"
]

pcos_ablation_df = df[
    df["source_dataset"] == "kaggle_pcos_extended_patient_dedup"
][
    pcos_ablation_features + ["pcos_target", "source_patient_id"]
].copy()

print("Dataset shape:", pcos_ablation_df.shape)

print("\nMissing values:")
print(pcos_ablation_df.isnull().sum())

print("\nTarget distribution:")
print(pcos_ablation_df["pcos_target"].value_counts())

print("\nFeature distributions:")
for col in pcos_ablation_features:
    print("\n", col)
    print(pcos_ablation_df[col].value_counts())

In [ ]:
X_ablation = pcos_ablation_df[pcos_ablation_features].copy()
y_ablation = pcos_ablation_df["pcos_target"].astype(int).copy()

X_ab_train, X_ab_temp, y_ab_train, y_ab_temp = train_test_split(
    X_ablation,
    y_ablation,
    test_size=0.30,
    stratify=y_ablation,
    random_state=42
)

X_ab_val, X_ab_test, y_ab_val, y_ab_test = train_test_split(
    X_ab_temp,
    y_ab_temp,
    test_size=0.50,
    stratify=y_ab_temp,
    random_state=42
)

print("Training:", X_ab_train.shape)
print("Validation:", X_ab_val.shape)
print("Testing:", X_ab_test.shape)

print("\nClass distribution:")
print("Train:", y_ab_train.value_counts().to_dict())
print("Validation:", y_ab_val.value_counts().to_dict())
print("Test:", y_ab_test.value_counts().to_dict())

In [ ]:
ablation_scaler = StandardScaler()

X_ab_train_scaled = ablation_scaler.fit_transform(X_ab_train)
X_ab_val_scaled = ablation_scaler.transform(X_ab_val)
X_ab_test_scaled = ablation_scaler.transform(X_ab_test)

print("Processed training:", X_ab_train_scaled.shape)
print("Processed validation:", X_ab_val_scaled.shape)
print("Processed testing:", X_ab_test_scaled.shape)

In [ ]:
def evaluate_ablation_model(model, X_test, y_test):
    probabilities = model.predict(X_test, verbose=0).ravel()
    predictions = (probabilities >= 0.5).astype(int)

    results = {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, probabilities)
    }

    cm = confusion_matrix(y_test, predictions)

    return results, cm, probabilities, predictions

In [ ]:
def build_ablation_dnn(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(64, activation="relu"),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dropout(0.2),
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
ablation_dnn = build_ablation_dnn(
    X_ab_train_scaled.shape[1]
)

ablation_early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history_ablation_dnn = ablation_dnn.fit(
    X_ab_train_scaled,
    y_ab_train,
    validation_data=(X_ab_val_scaled, y_ab_val),
    epochs=100,
    batch_size=16,
    callbacks=[ablation_early_stop],
    verbose=1
)

In [ ]:
dnn_ablation_results, dnn_ablation_cm, _, _ = evaluate_ablation_model(
    ablation_dnn,
    X_ab_test_scaled,
    y_ab_test
)

print("\nBASELINE DNN")
print("=" * 40)

for metric, value in dnn_ablation_results.items():
    print(f"{metric}: {value * 100:.2f}%")

print("\nConfusion Matrix:")
print(dnn_ablation_cm)

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

ablation_df = pcos_ablation_df[
    ["regular_exercise", "fast_food", "pcos_target"]
].copy()

ablation_df["lifestyle_interaction"] = (
    ablation_df["regular_exercise"] * ablation_df["fast_food"]
)

features_exp2 = [
    "regular_exercise",
    "fast_food",
    "lifestyle_interaction"
]

X = ablation_df[features_exp2].astype(float)
y = ablation_df["pcos_target"].astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

scaler_exp2 = StandardScaler()

X_train_scaled = scaler_exp2.fit_transform(X_train)
X_val_scaled = scaler_exp2.transform(X_val)
X_test_scaled = scaler_exp2.transform(X_test)

tf.random.set_seed(42)
np.random.seed(42)

model_exp2 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model_exp2.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history_exp2 = model_exp2.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=16,
    callbacks=[early_stopping],
    verbose=0
)

y_prob_exp2 = model_exp2.predict(
    X_test_scaled,
    verbose=0
).ravel()

y_pred_exp2 = (y_prob_exp2 >= 0.5).astype(int)

print("DNN + LIFESTYLE FEATURE ENGINEERING")
print(f"Accuracy: {accuracy_score(y_test, y_pred_exp2) * 100:.2f}%")
print(f"Precision: {precision_score(y_test, y_pred_exp2, zero_division=0) * 100:.2f}%")
print(f"Recall: {recall_score(y_test, y_pred_exp2, zero_division=0) * 100:.2f}%")
print(f"F1: {f1_score(y_test, y_pred_exp2, zero_division=0) * 100:.2f}%")
print(f"ROC_AUC: {roc_auc_score(y_test, y_prob_exp2) * 100:.2f}%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_exp2))

In [ ]:
import numpy as np
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

tf.random.set_seed(42)
np.random.seed(42)

input_dim = X_train_scaled.shape[1]

inputs = tf.keras.Input(shape=(input_dim,))

attention_scores = tf.keras.layers.Dense(
    input_dim,
    activation="softmax",
    name="attention_weights"
)(inputs)

weighted_features = tf.keras.layers.Multiply()(
    [inputs, attention_scores]
)

x = tf.keras.layers.Dense(
    64,
    activation="relu"
)(weighted_features)

x = tf.keras.layers.Dropout(0.2)(x)

x = tf.keras.layers.Dense(
    32,
    activation="relu"
)(x)

x = tf.keras.layers.Dropout(0.2)(x)

outputs = tf.keras.layers.Dense(
    1,
    activation="sigmoid"
)(x)

model_exp3 = tf.keras.Model(
    inputs=inputs,
    outputs=outputs
)

model_exp3.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping_exp3 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history_exp3 = model_exp3.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=16,
    callbacks=[early_stopping_exp3],
    verbose=0
)

y_prob_exp3 = model_exp3.predict(
    X_test_scaled,
    verbose=0
).ravel()

y_pred_exp3 = (y_prob_exp3 >= 0.5).astype(int)

print("DNN + LIFESTYLE ATTENTION")
print("=" * 40)

print(
    f"Accuracy: "
    f"{accuracy_score(y_test, y_pred_exp3) * 100:.2f}%"
)

print(
    f"Precision: "
    f"{precision_score(y_test, y_pred_exp3, zero_division=0) * 100:.2f}%"
)

print(
    f"Recall: "
    f"{recall_score(y_test, y_pred_exp3, zero_division=0) * 100:.2f}%"
)

print(
    f"F1: "
    f"{f1_score(y_test, y_pred_exp3, zero_division=0) * 100:.2f}%"
)

print(
    f"ROC_AUC: "
    f"{roc_auc_score(y_test, y_prob_exp3) * 100:.2f}%"
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_exp3))

In [ ]:
attention_model = tf.keras.Model(
    inputs=model_exp3.input,
    outputs=model_exp3.get_layer("attention_weights").output
)

attention_values = attention_model.predict(
    X_test_scaled,
    verbose=0
)

mean_attention = attention_values.mean(axis=0)

attention_df = pd.DataFrame({
    "Feature": features_exp2,
    "Attention": mean_attention
})

attention_df = attention_df.sort_values(
    "Attention",
    ascending=False
).reset_index(drop=True)

print("\nATTENTION WEIGHTS")
print("=" * 40)
print(attention_df)

In [ ]:
ablation_results = pd.DataFrame([
    {
        "Experiment": "Baseline DNN",
        "Accuracy": 65.57,
        "Precision": 58.33,
        "Recall": 77.78,
        "F1": 66.67,
        "ROC_AUC": 70.70
    },
    {
        "Experiment": "DNN + Feature Engineering",
        "Accuracy": 65.57,
        "Precision": 58.33,
        "Recall": 77.78,
        "F1": 66.67,
        "ROC_AUC": 67.97
    },
    {
        "Experiment": "Full LANN",
        "Accuracy": 65.57,
        "Precision": 58.33,
        "Recall": 77.78,
        "F1": 66.67,
        "ROC_AUC": 65.47
    }
])

ablation_results.to_csv(
    "PCOS_Lifestyle_Ablation_Results.csv",
    index=False
)

print(ablation_results)
print("\nSaved as: PCOS_Lifestyle_Ablation_Results.csv")

In [ ]:
model_exp3.save("PCOS_LANN_Model.keras")

import joblib

joblib.dump(
    scaler_exp2,
    "PCOS_LANN_Scaler.pkl"
)

print("LANN model saved.")
print("Scaler saved.")

In [ ]:
app_code = '''
import streamlit as st
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib

st.set_page_config(
    page_title="PCOS Lifestyle Screening",
    page_icon="🩺",
    layout="centered"
)

st.title("AI-Based PCOS Lifestyle Screening")
st.write(
    "Non-invasive lifestyle-based screening prototype "
    "using a Lifestyle Attention Neural Network."
)

model = tf.keras.models.load_model("PCOS_LANN_Model.keras")
scaler = joblib.load("PCOS_LANN_Scaler.pkl")

st.subheader("Patient Lifestyle Information")

regular_exercise = st.selectbox(
    "Regular Exercise",
    ["No", "Yes"]
)

fast_food = st.selectbox(
    "Frequent Fast Food Consumption",
    ["No", "Yes"]
)

if st.button("Predict"):

    exercise_value = 1 if regular_exercise == "Yes" else 0
    fast_food_value = 1 if fast_food == "Yes" else 0

    lifestyle_interaction = (
        exercise_value * fast_food_value
    )

    input_data = np.array([[
        exercise_value,
        fast_food_value,
        lifestyle_interaction
    ]], dtype=float)

    input_scaled = scaler.transform(input_data)

    probability = float(
        model.predict(input_scaled, verbose=0)[0][0]
    )

    prediction = (
        "PCOS Positive Screening"
        if probability >= 0.5
        else "PCOS Negative Screening"
    )

    st.subheader("Prediction Result")

    st.write(
        f"Prediction Probability: {probability * 100:.2f}%"
    )

    if probability >= 0.5:
        st.error(prediction)
    else:
        st.success(prediction)

    st.info(
        "This prototype is intended for academic screening "
        "demonstration only and is not a medical diagnosis."
    )
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py created successfully.")

In [ ]:
import matplotlib.pyplot as plt

metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC"
]

for metric in metrics:
    plt.figure(figsize=(9, 5))

    plt.bar(
        ablation_results["Experiment"],
        ablation_results[metric]
    )

    plt.title(f"PCOS Lifestyle Ablation - {metric}")
    plt.ylabel("Percentage")
    plt.xticks(rotation=15)
    plt.ylim(0, 100)
    plt.tight_layout()

    plt.savefig(
        f"PCOS_Ablation_{metric}.png",
        dpi=300
    )

    plt.show()

In [ ]:
import matplotlib.pyplot as plt

plot_df = ablation_results.set_index("Experiment")[
    ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]
]

plot_df.plot(
    kind="bar",
    figsize=(11, 6)
)

plt.title("PCOS Lifestyle Feature Ablation Study")
plt.ylabel("Performance (%)")
plt.xlabel("Experiment")
plt.xticks(rotation=15)
plt.ylim(0, 100)
plt.legend(title="Metric")
plt.tight_layout()

plt.savefig(
    "PCOS_Lifestyle_Ablation_Comparison.png",
    dpi=300
)

plt.show()

In [ ]:
final_results = pd.DataFrame([
    ["Diabetes", "Logistic Regression", 75.65, 75.44, 75.44, 75.44, 84.39],
    ["Diabetes", "Decision Tree", 71.30, 81.58, 54.39, 65.26, 75.48],
    ["Diabetes", "Random Forest", 74.78, 72.58, 78.95, 75.63, 85.66],
    ["Diabetes", "XGBoost", 73.04, 70.31, 78.95, 74.38, 83.85],
    ["Diabetes", "Standard DNN", 76.52, 75.86, 77.19, 76.52, 86.27],
    ["Diabetes", "LANN", 73.91, 72.88, 75.44, 74.14, 82.21],

    ["PCOS", "Logistic Regression", 61.74, 57.65, 85.96, 69.01, 57.99],
    ["PCOS", "Decision Tree", 57.39, 55.56, 70.18, 62.02, 58.73],
    ["PCOS", "Random Forest", 59.13, 57.14, 70.18, 62.99, 59.71],
    ["PCOS", "XGBoost", 63.48, 60.56, 75.44, 67.19, 66.15],
    ["PCOS", "Standard DNN", 61.74, 58.67, 77.19, 66.67, 58.56],
    ["PCOS", "LANN", 60.00, 56.18, 87.72, 68.49, 62.39],

    ["Thyroid", "Logistic Regression", 52.17, 51.92, 47.37, 49.54, 51.51],
    ["Thyroid", "Decision Tree", 47.83, 47.76, 56.14, 51.61, 50.03],
    ["Thyroid", "Random Forest", 49.57, 49.21, 54.39, 51.67, 55.11],
    ["Thyroid", "XGBoost", 49.57, 49.09, 47.37, 48.21, 53.71],
    ["Thyroid", "Standard DNN", 46.09, 45.76, 47.37, 46.55, 50.24],
    ["Thyroid", "LANN", 54.78, 54.39, 54.39, 54.39, 52.77]
], columns=[
    "Disease",
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC"
])

final_results.to_csv(
    "Final_Disease_Model_Results.csv",
    index=False
)

print(final_results.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

for disease in ["Diabetes", "PCOS", "Thyroid"]:

    disease_data = final_results[
        final_results["Disease"] == disease
    ]

    plt.figure(figsize=(10, 6))

    plt.bar(
        disease_data["Model"],
        disease_data["Accuracy"]
    )

    plt.title(f"{disease} - Model Accuracy Comparison")
    plt.ylabel("Accuracy (%)")
    plt.xlabel("Model")
    plt.xticks(rotation=30)
    plt.ylim(0, 100)
    plt.tight_layout()

    plt.savefig(
        f"{disease}_Accuracy_Comparison.png",
        dpi=300
    )

    plt.show()

In [ ]:
attention_plot_df = pd.DataFrame({
    "Feature": [
        "fast_food",
        "regular_exercise",
        "lifestyle_interaction"
    ],
    "Attention": [
        0.348692,
        0.329236,
        0.322071
    ]
})

plt.figure(figsize=(8, 5))

plt.bar(
    attention_plot_df["Feature"],
    attention_plot_df["Attention"]
)

plt.title("PCOS Lifestyle Feature Attention Weights")
plt.xlabel("Feature")
plt.ylabel("Mean Attention Weight")
plt.ylim(0, 0.4)
plt.tight_layout()

plt.savefig(
    "PCOS_Lifestyle_Attention.png",
    dpi=300
)

plt.show()

In [ ]:
model_exp3.save("PCOS_LANN_Model.keras")

import joblib

joblib.dump(
    scaler_exp2,
    "PCOS_LANN_Scaler.pkl"
)

print("Model and scaler saved successfully.")

In [ ]:
app_code = '''
import streamlit as st
import numpy as np
import tensorflow as tf
import joblib

st.set_page_config(
    page_title="PCOS Lifestyle Screening",
    page_icon="🩺"
)

st.title("AI-Based PCOS Lifestyle Screening")

st.write(
    "Non-invasive lifestyle-based screening prototype "
    "using a Lifestyle Attention Neural Network."
)

model = tf.keras.models.load_model("PCOS_LANN_Model.keras")
scaler = joblib.load("PCOS_LANN_Scaler.pkl")

st.subheader("Enter Lifestyle Information")

regular_exercise = st.selectbox(
    "Do you exercise regularly?",
    ["No", "Yes"]
)

fast_food = st.selectbox(
    "Do you frequently consume fast food?",
    ["No", "Yes"]
)

if st.button("Predict"):

    exercise_value = 1 if regular_exercise == "Yes" else 0
    fast_food_value = 1 if fast_food == "Yes" else 0

    lifestyle_interaction = (
        exercise_value * fast_food_value
    )

    input_data = np.array([[
        exercise_value,
        fast_food_value,
        lifestyle_interaction
    ]], dtype=float)

    input_scaled = scaler.transform(input_data)

    probability = float(
        model.predict(input_scaled, verbose=0)[0][0]
    )

    prediction = (
        "PCOS Positive Screening"
        if probability >= 0.5
        else "PCOS Negative Screening"
    )

    st.subheader("Prediction Result")

    st.write(
        f"Prediction Probability: {probability * 100:.2f}%"
    )

    if probability >= 0.5:
        st.error(prediction)
    else:
        st.success(prediction)

    st.info(
        "This is an academic screening prototype and "
        "not a medical diagnosis."
    )
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py created successfully.")

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("TensorFlow:", tf.__version__)

In [ ]:
df = pd.read_csv("balanced_multidisease_model_dataset.csv")

print("Dataset shape:", df.shape)
print(df.columns.tolist())

In [ ]:
pcos_df = df[df["pcos_target"].notna()].copy()

print("PCOS dataset shape:", pcos_df.shape)

print(
    pcos_df["pcos_target"].value_counts()
)

In [ ]:
features = [
    "age",
    "bmi",
    "height_cm",
    "weight_kg",
    "waist_hip_ratio",
    "systolic_bp",
    "diastolic_bp",
    "regular_exercise",
    "fast_food"
]

target = "pcos_target"

X = pcos_df[features].copy()
y = pcos_df[target].astype(int)

print("Features:")
print(features)

print("\nShape:", X.shape)

print("\nMissing values:")
print(X.isna().sum())

print("\nTarget:")
print(y.value_counts())

In [ ]:
X.loc[(X["age"] < 18) | (X["age"] > 100), "age"] = np.nan

X.loc[
    (X["bmi"] <= 10) | (X["bmi"] > 80),
    "bmi"
] = np.nan

X.loc[
    (X["height_cm"] < 100) | (X["height_cm"] > 250),
    "height_cm"
] = np.nan

X.loc[
    (X["weight_kg"] <= 20) | (X["weight_kg"] > 300),
    "weight_kg"
] = np.nan

X.loc[
    (X["waist_hip_ratio"] <= 0) | (X["waist_hip_ratio"] > 2),
    "waist_hip_ratio"
] = np.nan

X.loc[
    (X["systolic_bp"] < 70) | (X["systolic_bp"] > 250),
    "systolic_bp"
] = np.nan

X.loc[
    (X["diastolic_bp"] < 40) | (X["diastolic_bp"] > 150),
    "diastolic_bp"
] = np.nan

print("Missing values after cleaning:")
print(X.isna().sum())

In [ ]:
print("Regular exercise:")
print(X["regular_exercise"].value_counts(dropna=False))

print("\nFast food:")
print(X["fast_food"].value_counts(dropna=False))

In [ ]:
X["regular_exercise"] = pd.to_numeric(
    X["regular_exercise"],
    errors="coerce"
)

X["fast_food"] = pd.to_numeric(
    X["fast_food"],
    errors="coerce"
)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

In [ ]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

In [ ]:
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

print("\nTarget distributions:")

print("Train:")
print(y_train.value_counts())

print("\nValidation:")
print(y_val.value_counts())

print("\nTest:")
print(y_test.value_counts())

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import joblib
import os

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

tf.keras.backend.clear_session()

print("PCOS LANN - 9 FEATURE IMPLEMENTATION")

features = [
    "age",
    "bmi",
    "height_cm",
    "weight_kg",
    "waist_hip_ratio",
    "systolic_bp",
    "diastolic_bp",
    "regular_exercise",
    "fast_food"
]

print("\nFeatures:")
for i, feature in enumerate(features, 1):
    print(i, feature)

print("\nDataset sizes:")
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

for column in features:
    X_train[column] = pd.to_numeric(
        X_train[column],
        errors="coerce"
    )

    X_val[column] = pd.to_numeric(
        X_val[column],
        errors="coerce"
    )

    X_test[column] = pd.to_numeric(
        X_test[column],
        errors="coerce"
    )

def clean_pcos_data(data):

    data = data.copy()

    data.loc[
        (data["age"] < 18) |
        (data["age"] > 100),
        "age"
    ] = np.nan

    data.loc[
        (data["bmi"] <= 10) |
        (data["bmi"] > 80),
        "bmi"
    ] = np.nan

    data.loc[
        (data["height_cm"] < 100) |
        (data["height_cm"] > 250),
        "height_cm"
    ] = np.nan

    data.loc[
        (data["weight_kg"] <= 20) |
        (data["weight_kg"] > 300),
        "weight_kg"
    ] = np.nan

    data.loc[
        (data["waist_hip_ratio"] <= 0) |
        (data["waist_hip_ratio"] > 2),
        "waist_hip_ratio"
    ] = np.nan

    data.loc[
        (data["systolic_bp"] < 70) |
        (data["systolic_bp"] > 250),
        "systolic_bp"
    ] = np.nan

    data.loc[
        (data["diastolic_bp"] < 40) |
        (data["diastolic_bp"] > 150),
        "diastolic_bp"
    ] = np.nan

    data.loc[
        ~data["regular_exercise"].isin([0, 1]),
        "regular_exercise"
    ] = np.nan

    data.loc[
        ~data["fast_food"].isin([0, 1]),
        "fast_food"
    ] = np.nan

    return data

X_train = clean_pcos_data(X_train)
X_val = clean_pcos_data(X_val)
X_test = clean_pcos_data(X_test)

print("\nData cleaning completed.")

preprocessor = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

X_train_processed = preprocessor.fit_transform(
    X_train[features]
)

X_val_processed = preprocessor.transform(
    X_val[features]
)

X_test_processed = preprocessor.transform(
    X_test[features]
)

print("\nPreprocessing completed.")
print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

input_dim = len(features)

inputs = tf.keras.Input(
    shape=(input_dim,),
    name="input_features"
)

attention_scores = tf.keras.layers.Dense(
    input_dim,
    activation="softmax",
    name="attention_weights"
)(inputs)

weighted_features = tf.keras.layers.Multiply(
    name="weighted_features"
)(
    [inputs, attention_scores]
)

x = tf.keras.layers.Dense(
    64,
    activation="relu",
    name="dense_64"
)(weighted_features)

x = tf.keras.layers.Dropout(
    0.20,
    name="dropout_1"
)(x)

x = tf.keras.layers.Dense(
    32,
    activation="relu",
    name="dense_32"
)(x)

x = tf.keras.layers.Dropout(
    0.20,
    name="dropout_2"
)(x)

outputs = tf.keras.layers.Dense(
    1,
    activation="sigmoid",
    name="pcos_output"
)(x)

lann_model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="PCOS_LANN_9Feature"
)

lann_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\nMODEL ARCHITECTURE")
lann_model.summary()

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

print("\nTRAINING")

history = lann_model.fit(
    X_train_processed,
    y_train,
    validation_data=(
        X_val_processed,
        y_val
    ),
    epochs=100,
    batch_size=16,
    callbacks=[early_stopping],
    verbose=1
)

print(
    "\nTraining stopped after",
    len(history.history["loss"]),
    "epochs."
)

val_probabilities = lann_model.predict(
    X_val_processed,
    verbose=0
).ravel()

val_predictions = (
    val_probabilities >= 0.5
).astype(int)

val_accuracy = accuracy_score(
    y_val,
    val_predictions
)

val_precision = precision_score(
    y_val,
    val_predictions,
    zero_division=0
)

val_recall = recall_score(
    y_val,
    val_predictions,
    zero_division=0
)

val_f1 = f1_score(
    y_val,
    val_predictions,
    zero_division=0
)

val_roc_auc = roc_auc_score(
    y_val,
    val_probabilities
)

val_cm = confusion_matrix(
    y_val,
    val_predictions
)

print("\nVALIDATION RESULTS")
print(f"Accuracy : {val_accuracy * 100:.2f}%")
print(f"Precision: {val_precision * 100:.2f}%")
print(f"Recall   : {val_recall * 100:.2f}%")
print(f"F1 Score : {val_f1 * 100:.2f}%")
print(f"ROC-AUC  : {val_roc_auc * 100:.2f}%")

print("\nValidation Confusion Matrix:")
print(val_cm)

test_probabilities = lann_model.predict(
    X_test_processed,
    verbose=0
).ravel()

test_predictions = (
    test_probabilities >= 0.5
).astype(int)

test_accuracy = accuracy_score(
    y_test,
    test_predictions
)

test_precision = precision_score(
    y_test,
    test_predictions,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    test_predictions,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    test_predictions,
    zero_division=0
)

test_roc_auc = roc_auc_score(
    y_test,
    test_probabilities
)

test_cm = confusion_matrix(
    y_test,
    test_predictions
)

print("\nFINAL TEST RESULTS")
print(f"Accuracy : {test_accuracy * 100:.2f}%")
print(f"Precision: {test_precision * 100:.2f}%")
print(f"Recall   : {test_recall * 100:.2f}%")
print(f"F1 Score : {test_f1 * 100:.2f}%")
print(f"ROC-AUC  : {test_roc_auc * 100:.2f}%")

print("\nTest Confusion Matrix:")
print(test_cm)

results = pd.DataFrame([
    {
        "Disease": "PCOS",
        "Model": "LANN_9Feature",
        "Accuracy": test_accuracy * 100,
        "Precision": test_precision * 100,
        "Recall": test_recall * 100,
        "F1": test_f1 * 100,
        "ROC_AUC": test_roc_auc * 100
    }
])

results.to_csv(
    "PCOS_LANN_9Feature_Test_Results.csv",
    index=False
)

attention_model = tf.keras.Model(
    inputs=lann_model.input,
    outputs=lann_model.get_layer(
        "attention_weights"
    ).output
)

attention_values = attention_model.predict(
    X_test_processed,
    verbose=0
)

mean_attention = attention_values.mean(
    axis=0
)

attention_df = pd.DataFrame({
    "Feature": features,
    "Attention": mean_attention
})

attention_df = attention_df.sort_values(
    "Attention",
    ascending=False
)

print("\nLANN ATTENTION WEIGHTS")
print(
    attention_df.to_string(
        index=False
    )
)

attention_df.to_csv(
    "PCOS_LANN_9Feature_Attention.csv",
    index=False
)

model_filename = "PCOS_LANN_9Feature_Model.keras"

preprocessor_filename = "PCOS_LANN_9Feature_Preprocessor.pkl"

lann_model.save(
    model_filename
)

joblib.dump(
    preprocessor,
    preprocessor_filename
)

print("\nFILES SAVED")
print(model_filename)
print(preprocessor_filename)
print("PCOS_LANN_9Feature_Test_Results.csv")
print("PCOS_LANN_9Feature_Attention.csv")

try:

    from google.colab import files

    print("\nStarting downloads...")

    files.download(
        model_filename
    )

    files.download(
        preprocessor_filename
    )

    files.download(
        "PCOS_LANN_9Feature_Test_Results.csv"
    )

    files.download(
        "PCOS_LANN_9Feature_Attention.csv"
    )

except Exception as e:

    print(
        "\nAutomatic download unavailable."
    )

    print(
        "Files are saved in the current Colab directory."
    )

print("\nDONE")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve

plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("PCOS LANN Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("PCOS LANN Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(6, 5))
plt.imshow(test_cm)
plt.title("PCOS LANN Test Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0, 1], ["No PCOS", "PCOS"])
plt.yticks([0, 1], ["No PCOS", "PCOS"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, test_cm[i, j], ha="center", va="center")

plt.colorbar()
plt.show()

fpr, tpr, thresholds = roc_curve(
    y_test,
    test_probabilities
)

plt.figure(figsize=(7, 5))
plt.plot(
    fpr,
    tpr,
    label=f"ROC-AUC = {test_roc_auc:.3f}"
)
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("PCOS LANN ROC Curve")
plt.legend()
plt.grid(True)
plt.show()

attention_plot = attention_df.sort_values(
    "Attention",
    ascending=True
)

plt.figure(figsize=(8, 5))
plt.barh(
    attention_plot["Feature"],
    attention_plot["Attention"]
)
plt.xlabel("Mean Attention Weight")
plt.ylabel("Feature")
plt.title("PCOS LANN Feature Attention Weights")
plt.grid(axis="x")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import joblib

model = tf.keras.models.load_model(
    "PCOS_LANN_9Feature_Model.keras"
)

preprocessor = joblib.load(
    "PCOS_LANN_9Feature_Preprocessor.pkl"
)

features = [
    "age",
    "bmi",
    "height_cm",
    "weight_kg",
    "waist_hip_ratio",
    "systolic_bp",
    "diastolic_bp",
    "regular_exercise",
    "fast_food"
]

def predict_pcos(
    age,
    bmi,
    height_cm,
    weight_kg,
    waist_hip_ratio,
    systolic_bp,
    diastolic_bp,
    regular_exercise,
    fast_food
):

    input_data = pd.DataFrame([[
        age,
        bmi,
        height_cm,
        weight_kg,
        waist_hip_ratio,
        systolic_bp,
        diastolic_bp,
        regular_exercise,
        fast_food
    ]], columns=features)

    input_processed = preprocessor.transform(
        input_data
    )

    probability = model.predict(
        input_processed,
        verbose=0
    )[0][0]

    prediction = int(probability >= 0.5)

    return probability, prediction

print("PCOS LANN prediction system loaded successfully.")

In [ ]:
probability, prediction = predict_pcos(
    age=22,
    bmi=24.5,
    height_cm=160,
    weight_kg=63,
    waist_hip_ratio=0.80,
    systolic_bp=115,
    diastolic_bp=75,
    regular_exercise=1,
    fast_food=0
)

print("Predicted Probability:", round(probability * 100, 2), "%")

if prediction == 1:
    print("Prediction: Higher PCOS risk")
else:
    print("Prediction: Lower PCOS risk")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np

age_input = widgets.FloatText(
    description="Age:",
    value=22
)

height_input = widgets.FloatText(
    description="Height (cm):",
    value=160
)

weight_input = widgets.FloatText(
    description="Weight (kg):",
    value=60
)

waist_input = widgets.FloatText(
    description="Waist (cm):",
    value=70
)

hip_input = widgets.FloatText(
    description="Hip (cm):",
    value=90
)

systolic_input = widgets.FloatText(
    description="Systolic BP:",
    value=120
)

diastolic_input = widgets.FloatText(
    description="Diastolic BP:",
    value=80
)

exercise_input = widgets.Dropdown(
    options=[
        ("Yes", 1),
        ("No", 0)
    ],
    description="Exercise:"
)

fast_food_input = widgets.Dropdown(
    options=[
        ("Yes", 1),
        ("No", 0)
    ],
    description="Fast Food:"
)

predict_button = widgets.Button(
    description="Predict PCOS Risk",
    button_style="primary"
)

output = widgets.Output()

def make_prediction(button):

    with output:
        clear_output()

        age = age_input.value
        height = height_input.value
        weight = weight_input.value
        waist = waist_input.value
        hip = hip_input.value
        systolic = systolic_input.value
        diastolic = diastolic_input.value
        exercise = exercise_input.value
        fast_food = fast_food_input.value

        if age < 18 or age > 100:
            print("Please enter a valid age.")
            return

        if height < 100 or height > 250:
            print("Please enter a valid height.")
            return

        if weight <= 20 or weight > 300:
            print("Please enter a valid weight.")
            return

        if waist <= 0 or hip <= 0:
            print("Waist and hip measurements must be greater than 0.")
            return

        bmi = weight / ((height / 100) ** 2)
        waist_hip_ratio = waist / hip

        input_data = pd.DataFrame([[
            age,
            bmi,
            height,
            weight,
            waist_hip_ratio,
            systolic,
            diastolic,
            exercise,
            fast_food
        ]], columns=features)

        input_processed = preprocessor.transform(
            input_data
        )

        probability = model.predict(
            input_processed,
            verbose=0
        )[0][0]

        prediction = int(probability >= 0.5)

        print("PCOS LANN PREDICTION")
        print()
        print(f"BMI: {bmi:.2f}")
        print(f"Waist-Hip Ratio: {waist_hip_ratio:.2f}")
        print(f"Predicted Probability: {probability * 100:.2f}%")
        print()

        if prediction == 1:
            print("Prediction: Higher PCOS Risk")
        else:
            print("Prediction: Lower PCOS Risk")

        print()
        print("This is a machine-learning prediction and not a medical diagnosis.")

predict_button.on_click(make_prediction)

display(
    widgets.VBox([
        widgets.HTML("<h2>PCOS LANN Prediction System</h2>"),
        age_input,
        height_input,
        weight_input,
        waist_input,
        hip_input,
        systolic_input,
        diastolic_input,
        exercise_input,
        fast_food_input,
        predict_button,
        output
    ])
)